# ACM2 parameter extraction (ACM2 2.0.0)
Extract the five DC parameters of the ACM2 MOSFET model from transfer and output I–V curves of one device, NMOS or
PMOS, and export a ready-to-use `.model` card for `acm2.va`.

The notebook is self-contained: it needs a **CPU runtime** and an internet connection only to download the public
IHP measurements. No Google Drive, simulator, PDK installation or local files are needed.

**Data.** Choose any of the 76 measured IHP SG13G2 low-voltage devices (38 NMOS and 38 PMOS geometries,
W = 0.15–10 µm, L = 0.12–10 µm, 27 °C, V_S = V_B = 0), downloaded from IHP-Open-PDK at a pinned commit and checked
against SHA-256 hashes, or upload your own I–V files.

**Output.** For each device: the parameters VT0, ish, n, sigma and zeta, where **ish is the sheet specific current**:
the specific current of an instance is I_S = ish·w/l. Each card states the W and L at which it was extracted.
Also: per-curve errors, plots of currents and derivatives, and a ZIP with everything, including the exact code.

**Scope.** DC only, one device (W, L, temperature) per card. Capacitance, junction, noise, temperature-coefficient
and body-bias parameters are not extracted; the card leaves them at the model defaults.

**Start:** run the setup cells, choose the data, check the inventory, run the extraction, then review and download.

## 1. Prepare the runtime
Packages are installed only if a required minimum version is missing; the versions used are recorded with the
results. The hidden cells contain the implementation (open them to read the equations and the fitting method).

In [ ]:
import importlib.metadata as metadata
import subprocess
import sys
from pathlib import Path

requirements = {"numpy": ((1,24), "numpy>=1.24"), "scipy": ((1,15), "scipy>=1.15"),
                "pandas": ((1,5), "pandas>=1.5"), "matplotlib": ((3,7), "matplotlib>=3.7")}
missing = []
for name, (minimum, requirement) in requirements.items():
    try:
        installed = tuple(int(x) for x in metadata.version(name).split(".")[:2])
    except metadata.PackageNotFoundError:
        installed = (0,0)
    if installed < minimum:
        missing.append(requirement)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
WORK_DIR = Path("acm2_colab_work").resolve()
WORK_DIR.mkdir(exist_ok=True)
print("Workspace:", WORK_DIR)
for name in requirements:
    print(name, metadata.version(name))

In [ ]:
%%writefile acm2_colab_work/acm2_dc.py
"""ACM2 2.0.0 DC core (acm2.va) for one device at its extraction temperature, V_S = V_B = 0.

The equations are those of models/acm2.va: Algorithm 443 for the normalised charges, the cancellation-free
q_S - q_sat, and the specific current I_S = ish*W/L of the device. PMOS data are mapped to this NMOS frame by the
data loaders (voltages and currents negated), exactly as the Verilog-A code does for type = -1.
Model reference: D. G. Alves Neto et al., IEEE Access 12, 87420 (2024), doi:10.1109/ACCESS.2024.3417316.
"""
import numpy as np

# k and q of the thermal voltage $vt of the compiled model (OpenVAF: NIST 2010 values)
K_B = 1.3806488e-23
Q_E = 1.602176565e-19


def algorithm_443(z):
    """Vectorised translation of the Verilog-A algo_443 macro (principal Lambert W)."""
    z = np.asarray(z, dtype=float)
    tiny = np.finfo(float).tiny
    safe_z = np.maximum(z, tiny)
    result = np.empty_like(safe_z)
    low = safe_z < 0.7385
    if np.any(low):
        zl = safe_z[low]
        wn = (zl + (4.0/3.0)*zl**2)/(1.0 + (7.0/3.0)*zl + (5.0/6.0)*zl**2)
        zn = np.log(zl) - wn - np.log(np.maximum(wn, tiny))
        a = 2.0*(1.0 + wn)*(1.0 + wn + (2.0/3.0)*zn)
        result[low] = wn*(1.0 + zn/(1.0 + wn)*(a - zn)/(a - 2.0*zn))
    if np.any(~low):
        log_z = np.log(safe_z[~low])
        wn = np.maximum(log_z - 24.0*(log_z**2 + 2.0*log_z - 3.0)/(7.0*log_z**2 + 58.0*log_z + 127.0), tiny)
        zn = log_z - wn - np.log(wn)
        a = 2.0*(1.0 + wn)*(1.0 + wn + (2.0/3.0)*zn)
        result[~low] = wn*(1.0 + zn/(1.0 + wn)*(a - zn)/(a - 2.0*zn))
    return float(result) if result.ndim == 0 else np.maximum(result, 0.0)


def algorithm_443_from_log(log_z):
    """Algorithm 443 of exp(log_z) without exponential overflow."""
    return algorithm_443(np.exp(np.clip(log_z, -700.0, 700.0)))


PARAMETER_NAMES = ("vt0", "ish", "n", "sigma", "zeta")
# The optimizer searches log10 of the device specific current I_S = ish*W/L (same scale for every geometry);
# the reported parameter is ish = I_S*L/W.
VECTOR_NAMES = ("vt0", "log10_IS_device", "n", "sigma", "log10_zeta")
DEFAULT_BOUNDS = ((0.05, 0.90), (-10.0, -2.0), (1.0, 3.0), (0.0, 0.50), (-5.0, -0.05))


def decode(vector, W_um, L_um):
    """Optimizer vector -> card parameters of a device drawn at W_um x L_um (ish = I_S L/W)."""
    vt0, log_is, n, sigma, log_zeta = np.asarray(vector, dtype=float)
    return dict(zip(PARAMETER_NAMES, (float(vt0), float(10**log_is*L_um/W_um), float(n), float(sigma), float(10**log_zeta))))


def drain_current(vg, vd, parameters, W_um, L_um, temperature_K=300.15, m=1.0):
    """NMOS-frame drain current, V_S = V_B = 0, T = tref (the extraction temperature)."""
    p = parameters
    values = np.array([p[k] for k in PARAMETER_NAMES] + [W_um, L_um, temperature_K, m])
    if not np.all(np.isfinite(values)) or min(p["ish"], p["n"], W_um, L_um, temperature_K, m) <= 0 or p["zeta"] < 0:
        raise ValueError("Finite positive ish, n, W, L, temperature and m, and zeta >= 0 required.")
    vg, vd = np.broadcast_arrays(np.asarray(vg, float), np.asarray(vd, float))
    phit = K_B*temperature_K/Q_E
    specific = p["ish"]*W_um/L_um
    zeta = p["zeta"]
    vp = (vg - p["vt0"] + p["sigma"]*vd)/p["n"]
    q1 = np.asarray(algorithm_443_from_log(vp/phit + 1.0))
    # q1 - qsat in cancellation-free form, as in acm2.va (valid for zeta = 0)
    dqs = 2.0*q1/(1.0 + zeta + np.sqrt((1.0 + zeta)**2 + 2.0*zeta*q1))
    qsat = q1 - dqs
    q2 = np.asarray(algorithm_443_from_log(np.log(np.maximum(dqs, np.finfo(float).tiny)) - vd/phit + dqs)) + qsat
    delta = q1 - q2
    return m*specific*(q2 + q1 + 2.0)*delta/(1.0 + np.sqrt((zeta*delta)**2 + 0.1**2))

In [ ]:
%%writefile acm2_colab_work/acm2_extraction.py
"""Portable ACM2 2.0.0 extraction: public IHP MDM data (NMOS and PMOS) or user I-V files.

Model: models/acm2.va (ACM2 2.0.0); Alves Neto et al., IEEE Access 2024, doi:10.1109/ACCESS.2024.3417316.
Extraction: current-only differential evolution, derivative-weighted differential evolution, bounded least squares.
Output: one .model card per device with the sheet specific current ish = I_S L/W and the W, L of the extraction.
PMOS data are negated on loading (NMOS frame) and exported with type = -1.
"""
from __future__ import annotations
from dataclasses import dataclass, asdict, field
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import io
import json
import platform
import re
import time
import urllib.request
import zipfile

import numpy as np
import pandas as pd
import scipy
from scipy.optimize import differential_evolution, least_squares
from scipy.signal import savgol_filter
import matplotlib
# Batch/vector rendering also works on machines without a desktop GUI.
# The notebook displays the saved PNGs explicitly.
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from acm2_dc import decode, drain_current, DEFAULT_BOUNDS, VECTOR_NAMES

CURRENT_FLOOR = 1e-13
CURRENT_WEIGHTS = {"idvg": .25, "log_idvg": .30, "idvd": .25,
                   "log_idvd": .20, "gm": 0., "gds": 0.}
FINAL_WEIGHTS = {"idvg": .20, "log_idvg": .20, "idvd": .20,
                "log_idvd": .10, "gm": .20, "gds": .10}
PROFILES = {"quick": (15, 20, 6), "standard": (75, 100, 8),
            "extended": (150, 250, 12)}


def sha256(data):
    return hashlib.sha256(data).hexdigest()


def write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False),
                          encoding="utf-8")


@dataclass
class Curve:
    curve_id: str
    sweep: str
    x: np.ndarray
    vg: np.ndarray
    vd: np.ndarray
    measured: np.ndarray
    split: str
    operators: dict
    derivatives: dict
    fixed_voltage: float


@dataclass
class Device:
    device_id: str
    W_um: float
    L_um: float
    temperature_C: float
    m: float
    curves: list
    polarity: str = "nmos"
    excluded_curves: list = field(default_factory=list)

    @property
    def temperature_K(self):
        return self.temperature_C + 273.15


def parse_mdm(text):
    """Read ICCAP numeric blocks; reject malformed data instead of skipping it."""
    blocks, inside, bias, columns, rows = [], False, {}, None, []
    for raw in text.splitlines():
        line = raw.strip()
        if line == "BEGIN_DB":
            inside, bias, columns, rows = True, {}, None, []
        elif line == "END_DB" and inside:
            if columns is None or not rows:
                raise ValueError("Incomplete MDM block.")
            blocks.append((bias, pd.DataFrame(rows, columns=columns)))
            inside = False
        elif inside and line:
            if line.startswith("ICCAP_VAR"):
                parts = line.split()
                bias[parts[1].lower()] = float(parts[2])
            elif line.startswith("#"):
                candidate = line[1:].lower().split()
                if candidate and candidate[0] in {"vg", "vd", "vb", "vs"}:
                    columns = candidate
            elif columns is not None:
                parts = line.split()
                if len(parts) != len(columns):
                    raise ValueError("Malformed MDM row.")
                rows.append([float(p) for p in parts])
    if inside or not blocks:
        raise ValueError("Truncated or empty MDM document.")
    return blocks


def ihp_dataset(spec, selected, cache_dir, holdout=True):
    """Download commit-pinned measurements; validate content hashes. PMOS: voltages and currents negated."""
    cache = Path(cache_dir)
    cache.mkdir(parents=True, exist_ok=True)
    records, sources = [], []
    available = {d["device_id"]: d for d in spec["devices"]}
    if not selected or any(k not in available for k in selected):
        raise ValueError("Select at least one known IHP device.")
    for device_id in selected:
        d = available[device_id]
        for item in d["files"]:
            path = cache / item["filename"]
            if not path.exists():
                request = urllib.request.Request(item["url"],
                    headers={"User-Agent": "ACM2-Colab-Extraction"})
                try:
                    with urllib.request.urlopen(request, timeout=60) as response:
                        data = response.read()
                except Exception as exc:
                    raise RuntimeError("IHP download failed. Retry this cell; "
                                       "completed downloads stay cached.") from exc
                if sha256(data) != item["sha256"]:
                    raise ValueError("IHP content hash mismatch: " + item["filename"])
                path.write_bytes(data)
            data = path.read_bytes()
            if sha256(data) != item["sha256"]:
                raise ValueError("Cached IHP file changed: " + str(path))
            sources.append(dict(item, device_id=device_id))
            text = data.decode("utf-8")
            temp = re.search(r'\bTEMP\s+"?\s*([+-]?\d+(?:\.\d+)?)', text)
            if temp is None or not np.isclose(float(temp.group(1)), d["temperature_C"]):
                raise ValueError("IHP temperature differs from the pinned metadata.")
            s = -1.0 if d.get("polarity", "nmos") == "pmos" else 1.0   # type = -1: V_eff = -V, I = -I_core
            for number, (bias, frame) in enumerate(parse_mdm(text)):
                if abs(bias.get("vb", 0)) > 1e-12 or abs(bias.get("vs", 0)) > 1e-12:
                    continue
                sweep = "idvg" if "vg" in frame.columns else "idvd"
                fixed = s*(bias["vd"] if sweep == "idvg" else bias["vg"])
                vg = s*frame["vg"].to_numpy() if sweep == "idvg" else np.full(len(frame), fixed)
                vd = s*frame["vd"].to_numpy() if sweep == "idvd" else np.full(len(frame), fixed)
                split = "validation" if holdout and sweep == "idvg" and np.isclose(fixed, .6) else "fit"
                curve_id = f'{Path(item["filename"]).stem}_{number}'
                for gate, drain, current in zip(vg, vd, s*frame["id"].to_numpy()):
                    records.append(dict(device_id=device_id, curve_id=curve_id,
                        sweep=sweep, vg_V=float(gate), vd_V=float(drain),
                        id_A=float(current), split=split, W_um=d["W_um"],
                        L_um=d["L_um"], temperature_C=d["temperature_C"], m=1.0,
                        polarity=d.get("polarity", "nmos")))
    return pd.DataFrame(records), sources


def user_dataset(manifest_path):
    """General delimited files plus a manifest defining voltage/current columns.

    Every input file must be beside the manifest. A transfer sweep specifies VD;
    an output sweep specifies VG. Source and bulk are grounded. Numeric columns
    use decimal points; optional scale factors convert their units into V/A.
    """
    manifest_path = Path(manifest_path)
    meta = pd.read_csv(manifest_path)
    required = {"device_id", "file", "sweep", "fixed_voltage_V", "x_column",
                "current_column", "W_um", "L_um", "temperature_C"}
    if not required.issubset(meta.columns):
        raise ValueError("Missing manifest columns: " + str(sorted(required - set(meta.columns))))
    records, sources = [], []
    for index, row in meta.iterrows():
        filename = str(row["file"])
        if Path(filename).name != filename or "/" in filename or "\\" in filename:
            raise ValueError("Use filenames only, all beside manifest.csv.")
        path = manifest_path.parent / filename
        data = path.read_bytes()
        skip = int(row.get("skiprows", 0)) if pd.notna(row.get("skiprows", 0)) else 0
        sep = row.get("separator", "auto")
        sep = "auto" if pd.isna(sep) else str(sep)
        separators = {"auto": None, "comma": ",", "semicolon": ";", "tab": "\t", "space": r"\s+"}
        if sep not in separators:
            raise ValueError("separator must be auto/comma/semicolon/tab/space.")
        frame = pd.read_csv(io.BytesIO(data), sep=separators[sep], engine="python",
                            skiprows=skip, comment="#")
        x = pd.to_numeric(frame[str(row["x_column"])], errors="raise").to_numpy(float)
        y = pd.to_numeric(frame[str(row["current_column"])], errors="raise").to_numpy(float)
        def option(name, default):
            value = row.get(name, default)
            return default if pd.isna(value) else value
        xs, ys = float(option("voltage_scale", 1)), float(option("current_scale", 1))
        if not np.isfinite(xs + ys) or xs <= 0 or ys <= 0:
            raise ValueError("Use positive finite unit scales; ID is the current into the drain.")
        polarity = str(option("polarity", "nmos")).lower()
        if polarity not in {"nmos", "pmos"}:
            raise ValueError("polarity must be nmos or pmos.")
        s = -1.0 if polarity == "pmos" else 1.0   # PMOS: measured (negative) V and I -> NMOS frame
        x, y = s * x * xs, s * y * ys
        sweep, fixed = str(row["sweep"]).lower(), s * float(row["fixed_voltage_V"])
        if sweep not in {"idvg", "idvd"}:
            raise ValueError("sweep must be idvg or idvd.")
        sources.append({"file": filename, "sha256": sha256(data),
                        "manifest_row": int(index), "evidence": "user supplied"})
        for voltage, current in zip(x, y):
            records.append(dict(device_id=str(row["device_id"]),
                curve_id=f"file_{index}_{Path(filename).stem}", sweep=sweep,
                vg_V=float(voltage if sweep == "idvg" else fixed),
                vd_V=float(fixed if sweep == "idvg" else voltage), id_A=float(current),
                split=str(option("split", "fit")), W_um=float(row["W_um"]),
                L_um=float(row["L_um"]), temperature_C=float(row["temperature_C"]),
                m=float(option("m", 1)), polarity=polarity))
    sources.append({"file": manifest_path.name,
                    "sha256": sha256(manifest_path.read_bytes()), "evidence": "user metadata"})
    return pd.DataFrame(records), sources


def prepare_devices(frame):
    required = ["W_um", "L_um", "temperature_C", "m", "vg_V", "vd_V", "id_A"]
    if frame.empty or not np.isfinite(frame[required].to_numpy(float)).all():
        raise ValueError("Data are empty or contain NaN/infinite/non-numeric values.")
    if "polarity" not in frame.columns:
        frame = frame.assign(polarity="nmos")
    devices = []
    for name, group in frame.groupby("device_id", sort=False):
        if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]*", str(name)):
            raise ValueError("device_id must contain letters/digits/underscore/dot/hyphen.")
        metadata = {}
        for key in required[:4]:
            values = group[key].unique()
            if len(values) != 1:
                raise ValueError(f"{name}: mixed {key}; give each geometry/temperature a separate ID.")
            metadata[key] = float(values[0])
        if min(metadata["W_um"], metadata["L_um"]) <= 0 or metadata["m"] < 1 or metadata["temperature_C"] <= -273.15:
            raise ValueError("Positive W/L, m>=1 and positive absolute temperature required.")
        if group["polarity"].nunique() != 1:
            raise ValueError(f"{name}: mixed polarity; give NMOS and PMOS separate IDs.")
        metadata["polarity"] = str(group["polarity"].iloc[0])
        curves, excluded = [], []
        for curve_id, points in group.groupby("curve_id", sort=False):
            if points["sweep"].nunique() != 1 or points["split"].nunique() != 1:
                raise ValueError("A curve must have one sweep and one fit/validation assignment.")
            sweep, split = points["sweep"].iloc[0], points["split"].iloc[0]
            if sweep not in {"idvg", "idvd"} or split not in {"fit", "validation"}:
                raise ValueError("Expected idvg/idvd and fit/validation.")
            varying, fixed = ("vg_V", "vd_V") if sweep == "idvg" else ("vd_V", "vg_V")
            points = points.sort_values(varying)
            x, y = points[varying].to_numpy(float), points["id_A"].to_numpy(float)
            fixed_values = points[fixed].to_numpy(float)
            if not np.allclose(fixed_values, fixed_values[0], rtol=0, atol=1e-12):
                raise ValueError(f"{curve_id}: fixed bias varies inside the curve.")
            if len(x) < 11 or not np.all(np.diff(x) > 0):
                raise ValueError(f"{curve_id}: need at least 11 distinct voltage points.")
            step = float(np.median(np.diff(x)))
            if not np.allclose(np.diff(x), step, rtol=1e-5, atol=1e-12):
                raise ValueError(f"{curve_id}: SG derivatives require a uniform grid; no silent interpolation.")
            vg, vd = points["vg_V"].to_numpy(float), points["vd_V"].to_numpy(float)
            if not np.any((vg > 1e-12) & (vd > 1e-12)):
                excluded.append({"device_id": str(name), "curve_id": str(curve_id),
                                 "reason": "No VG>0 and VDS>0 samples; raw rows retained."})
                continue
            if np.any(vd < -1e-12) or np.max(y[(vg > 0) & (vd > 0)], initial=0) <= 0:
                raise ValueError(f"{curve_id}: expected forward data (NMOS frame) and positive current; "
                                 "PMOS data need polarity = pmos.")
            # The matrix applies exactly the same SG operation to reference/model.
            orders = (1, 2, 3) if sweep == "idvg" else (1,)
            operators = {order: savgol_filter(np.eye(len(x)),
                window_length=9 if order == 1 else 11,
                polyorder=3 if order == 1 else 5, deriv=order, delta=step,
                axis=0, mode="interp") for order in orders}
            curves.append(Curve(str(curve_id), sweep, x, vg, vd, y, split, operators,
                                {o: mat @ y for o, mat in operators.items()}, float(fixed_values[0])))
        if {c.sweep for c in curves if c.split == "fit"} != {"idvg", "idvd"}:
            raise ValueError(f"{name}: fitting requires transfer AND output curves.")
        devices.append(Device(str(name), curves=curves, excluded_curves=excluded, **metadata))
    return devices


class Objective:
    def __init__(self, device, weights, current_floor=CURRENT_FLOOR):
        self.device, self.weights, self.floor = device, dict(weights), float(current_floor)
        if self.floor <= 0 or not np.isfinite(self.floor):
            raise ValueError("Current floor must be positive and finite.")
        if set(self.weights) != set(CURRENT_WEIGHTS) or any(
                not np.isfinite(w) or w < 0 for w in self.weights.values()) or sum(self.weights.values()) <= 0:
            raise ValueError("Weights must contain six finite nonnegative quantity weights.")
        self.curves = [c for c in device.curves if c.split == "fit"]
        self.vg = np.concatenate([c.vg for c in self.curves])
        self.vd = np.concatenate([c.vd for c in self.curves])
        self.offsets = np.cumsum([0] + [len(c.x) for c in self.curves])
        self.terms = []
        for i, curve in enumerate(self.curves):
            mask = (curve.vg > 1e-12) & (curve.vd > 1e-12)
            log_mask = mask & (curve.measured > self.floor)
            for name, order, valid in [(curve.sweep, 0, mask),
                ("log_" + curve.sweep, -1, log_mask),
                ("gm" if curve.sweep == "idvg" else "gds", 1, mask)]:
                if self.weights[name] <= 0 or not np.any(valid):
                    continue
                reference = (np.log10(curve.measured[valid]) if order == -1 else
                    (curve.measured[valid] if order == 0 else curve.derivatives[order][valid]))
                scale = 1.0 if order == -1 else max(
                    float(np.max(np.abs(reference))), self.floor if order == 0 else 1e-12)
                self.terms.append((i, name, order, valid, reference, scale))
        counts = {name: sum(term[1] == name for term in self.terms) for name in self.weights}
        self.scales = [np.sqrt(self.weights[name] / (counts[name] * len(ref))) / scale
                       for _, name, _, _, ref, scale in self.terms]
        self.size = sum(len(term[4]) for term in self.terms)
        if not self.size:
            raise ValueError("No samples remain in the objective.")

    def residual(self, vector):
        d = self.device
        y = drain_current(self.vg, self.vd, decode(vector, d.W_um, d.L_um), d.W_um, d.L_um,
                          d.temperature_K, d.m)
        blocks = []
        for term, factor in zip(self.terms, self.scales):
            i, _, order, valid, ref, _ = term
            local = y[self.offsets[i]:self.offsets[i+1]]
            prediction = (np.log10(np.maximum(local[valid], self.floor)) if order == -1 else
                (local[valid] if order == 0 else
                 (self.curves[i].operators[order] @ local)[valid]))
            blocks.append((prediction - ref) * factor)
        result = np.concatenate(blocks)
        if not np.isfinite(result).all():
            raise FloatingPointError("Nonfinite ACM2 objective.")
        return result

    def __call__(self, vector):
        try:
            r = self.residual(vector)
            return float(r @ r)
        except (ValueError, ArithmeticError):
            return 1e20


def fit_device(device, output_dir, profile="standard", seeds=(11,),
               bounds=DEFAULT_BOUNDS, weights=None, current_floor=CURRENT_FLOOR):
    """Two DE stages, then bounded least-squares polishing; retain best cost."""
    if profile not in PROFILES:
        raise ValueError("Unknown fitting profile.")
    bounds = np.asarray(bounds, float)
    if bounds.shape != (5, 2) or not np.isfinite(bounds).all() or np.any(bounds[:, 0] >= bounds[:, 1]):
        raise ValueError("Expected five finite, increasing parameter bounds.")
    if bounds[0, 0] < 0 or bounds[2, 0] < 1 or bounds[2, 1] > 3 or bounds[3, 0] < 0:
        raise ValueError("Bounds must respect the released model: VT0>=0, 1<=n<=3, sigma>=0.")
    if not seeds or any(int(s) != s or s < 0 for s in seeds):
        raise ValueError("Use at least one nonnegative integer seed.")
    folder = Path(output_dir) / device.device_id
    folder.mkdir(parents=True, exist_ok=True)
    first = Objective(device, CURRENT_WEIGHTS, current_floor)
    final = Objective(device, FINAL_WEIGHTS if weights is None else weights, current_floor)
    a, b, popsize = PROFILES[profile]
    runs = []
    for seed in seeds:
        started = time.perf_counter()
        print(f"{device.device_id}: seed {seed}, current-only DE ({a} generations)", flush=True)
        common = dict(bounds=bounds, popsize=popsize, tol=1e-6, atol=0.,
                      polish=False, updating="immediate", workers=1)
        current = differential_evolution(first, maxiter=a,
                    rng=np.random.default_rng(seed), **common)
        checkpoint = {"stage": "current_only", "seed": int(seed),
                      "parameters": decode(current.x, device.W_um, device.L_um), "vector": current.x.tolist(),
                      "success": bool(current.success), "message": str(current.message)}
        write_json(folder / f"checkpoint_seed_{seed}.json", checkpoint)
        print(f"{device.device_id}: derivative-weighted DE ({b} generations)", flush=True)
        weighted = differential_evolution(final, maxiter=b, x0=current.x,
                    rng=np.random.default_rng(seed + 100000), **common)
        start = min([current.x, weighted.x], key=final)
        polished = least_squares(final.residual, start, bounds=(bounds[:, 0], bounds[:, 1]),
                     x_scale="jac", max_nfev=500, ftol=1e-10, xtol=1e-10, gtol=1e-10)
        candidates = [("current_de", current.x), ("weighted_de", weighted.x),
                      ("least_squares", polished.x)]
        selected, vector = min(candidates, key=lambda item: final(item[1]))
        near = np.minimum((vector - bounds[:, 0]) / np.diff(bounds).ravel(),
                          (bounds[:, 1] - vector) / np.diff(bounds).ravel())
        result = dict(seed=int(seed), parameters=decode(vector, device.W_um, device.L_um), vector=vector.tolist(),
            objective=final(vector), selected_stage=selected,
            elapsed_s=time.perf_counter() - started,
            bounds_near=[VECTOR_NAMES[i] for i in np.flatnonzero(near < .01)],
            stages={name: {"success": bool(r.success), "message": str(r.message),
                          "nfev": int(r.nfev)} for name, r in
                    [("current_de", current), ("weighted_de", weighted), ("least_squares", polished)]})
        write_json(folder / f"fit_seed_{seed}.json", result)
        runs.append(result)
        print(f"  final cost {result['objective']:.6g}; bound flags: {result['bounds_near']}", flush=True)
    best = min(runs, key=lambda r: r["objective"])
    write_json(folder / "best_fit.json", best)
    return best, runs


def evaluate(device, parameters, current_floor=CURRENT_FLOOR):
    metrics, samples = [], []
    for c in device.curves:
        y = drain_current(c.vg, c.vd, parameters, device.W_um, device.L_um, device.temperature_K, device.m)
        derivatives = {o: mat @ y for o, mat in c.operators.items()}
        mask = (c.vg > 1e-12) & (c.vd > 1e-12)
        for quantity, ref, pred, floor in [(c.sweep, c.measured, y, current_floor)] + [
                (("gm" if o == 1 else f"gm{o}") if c.sweep == "idvg" else "gds",
                 c.derivatives[o], derivatives[o], 1e-12) for o in c.operators]:
            scale = max(float(np.max(np.abs(ref[mask]))), floor)
            metrics.append(dict(device_id=device.device_id, curve_id=c.curve_id,
                split=c.split, quantity=quantity, points=int(mask.sum()),
                max_normalized_RMSE_pct=float(100*np.sqrt(np.mean((pred[mask]-ref[mask])**2))/scale),
                normalization=scale, derivative="none" if quantity in {"idvg","idvd"} else "matched SG"))
        rows = pd.DataFrame(dict(device_id=device.device_id, curve_id=c.curve_id,
            sweep=c.sweep, split=c.split, vg_V=c.vg, vd_V=c.vd,
            measured_id_A=c.measured, acm2_id_A=y, scored=mask))
        for order in c.operators:
            key = ("gm" if order == 1 else f"gm{order}") if c.sweep == "idvg" else "gds"
            rows["measured_" + key] = c.derivatives[order]
            rows["acm2_" + key] = derivatives[order]
        samples.append(rows)
    return pd.DataFrame(metrics), pd.concat(samples, ignore_index=True)


def comparison_figure(device, parameters):
    plt.rcParams.update({"font.family": "serif", "font.serif": ["STIXGeneral", "DejaVu Serif"],
                         "mathtext.fontset": "stix", "font.size": 10, "axes.labelsize": 11,
                         "axes.linewidth": 1., "lines.linewidth": 1.4, "savefig.bbox": "tight"})
    fig, axes = plt.subplots(2, 3, figsize=(10.5, 6.5), constrained_layout=True)
    colors = plt.get_cmap("tab10")
    for index, c in enumerate(device.curves):
        y = drain_current(c.vg, c.vd, parameters, device.W_um, device.L_um, device.temperature_K, device.m)
        label = f"{'VD' if c.sweep=='idvg' else 'VG'}={c.fixed_voltage:g} V"
        if c.split == "validation":
            label += " (held out)"
        color = colors(index % 10)
        groups = [(axes[0,0], c.measured, y, "log")] if c.sweep == "idvg" else [
            (axes[0,1], c.measured*1e3, y*1e3, "linear")]
        if c.sweep == "idvg":
            groups += [(axes[0,2], c.derivatives[1]*1e3, c.operators[1] @ y*1e3, "linear"),
                       (axes[1,1], c.derivatives[2], c.operators[2] @ y, "linear"),
                       (axes[1,2], c.derivatives[3], c.operators[3] @ y, "linear")]
        else:
            groups += [(axes[1,0], c.derivatives[1]*1e3, c.operators[1] @ y*1e3, "linear")]
        for ax, reference, prediction, scale in groups:
            ax.plot(c.x, np.where(reference > 0, reference, np.nan) if scale=="log" else reference,
                    "o", color=color, markersize=2.5, fillstyle="none", label=label)
            ax.plot(c.x, np.where(prediction > 0, prediction, np.nan) if scale=="log" else prediction,
                    "-", color=color)
            ax.set_yscale(scale)
    for ax, xlabel, ylabel in zip(axes.flat,
        [r"$V_{GB}$ (V)",r"$V_{DS}$ (V)",r"$V_{GB}$ (V)",r"$V_{DS}$ (V)",r"$V_{GB}$ (V)",r"$V_{GB}$ (V)"],
        [r"$I_D$ (A)",r"$I_D$ (mA)",r"$g_m$ (mS)",r"$g_{ds}$ (mS)",
         r"$g_{m2}$ (A/V$^2$)",r"$g_{m3}$ (A/V$^3$)"]):
        ax.set(xlabel=xlabel, ylabel=ylabel)
        ax.grid(True, which="major", alpha=.25)
        ax.tick_params(direction="in", top=True, right=True)
    # Bias legend only on transfer panel; output biases are recorded in CSV.
    axes[0,0].legend(fontsize=8)
    frame_note = " (PMOS shown in the NMOS frame: -V, -I)" if device.polarity == "pmos" else ""
    fig.suptitle(f"{device.device_id}: {device.polarity.upper()}, W={device.W_um:g} um, L={device.L_um:g} um, "
                 f"T={device.temperature_C:g} C{frame_note}\nMarkers: input data; lines: ACM2 2.0.0; "
                 "gm2/gm3: validation only (no factorial scaling)", fontsize=11)
    return fig


def model_name(device):
    base = re.sub(r"[^A-Za-z0-9_]", "_", device.device_id)
    return f"acm2{device.polarity[0]}_{base}"


def card_text(device, parameters):
    """ACM2 2.0.0 .model card: DC parameters with ish = I_S L/W and the W, L of the extraction."""
    name = model_name(device)
    lines = [f"* ACM2 2.0.0 DC card: {device.device_id} ({device.polarity.upper()})",
             f"* Extracted at W = {device.W_um:g} um, L = {device.L_um:g} um, T = {device.temperature_C:g} C, m = {device.m:g}.",
             "* The specific current of each instance is ish*w/l; use the card at the extraction L (and W, or",
             "* W >= 2 um for wide devices) and set .temp to the extraction temperature (tref below).",
             "* DC parameters only: capacitance, junction, gate-resistance, noise, temperature and mismatch",
             "* parameters stay at the model defaults and were not extracted.",
             "* Model: acm2.va (ACM2 2.0.0), compiled with OpenVAF (OSDI). Example instance:",
             f"*   N1 d g s b {name} w={device.W_um:g}u l={device.L_um:g}u",
             f".model {name} acm2_va",
             f"+ type={-1 if device.polarity == 'pmos' else 1}"]
    lines += [f"+ {k}={v:.12g}" for k, v in parameters.items()]
    lines += [f"+ tref={device.temperature_K:.12g}", ""]
    return "\n".join(lines)


def run_extraction(frame, sources, output_dir, profile="standard", seeds=(11,),
                   bounds=DEFAULT_BOUNDS, weights=None, current_floor=CURRENT_FLOOR):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=False)
    devices = prepare_devices(frame)
    exclusions = [item for d in devices for item in d.excluded_curves]
    write_json(output_dir / "excluded_curves.json", exclusions)
    if exclusions:
        print(f"{len(exclusions)} curves have no positive-bias fit samples; "
              "see excluded_curves.json. Raw input is retained.", flush=True)
    frame.to_csv(output_dir / "input_data_normalized.csv", index=False)
    config = dict(created_utc=datetime.now(timezone.utc).isoformat(),
        evidence="IHP measurements or user input, as identified by input_sources",
        profile=profile, seeds=list(seeds), bounds=np.asarray(bounds).tolist(),
        vector_names=VECTOR_NAMES, current_weights=CURRENT_WEIGHTS,
        final_weights=FINAL_WEIGHTS if weights is None else weights, current_floor_A=current_floor,
        derivative_settings={"gm": [9,3], "gds": [9,3], "gm2": [11,5], "gm3": [11,5]},
        derivative_edge_mode="interp", gm2_gm3_fitted=False,
        objective="Weighted per-quantity, per-curve mean squared residual; linear residuals "
                  "max-normalized per curve; log10 residuals in decades.",
        score_mask="VG>0 and VDS>0; full endpoint samples retained for differentiation",
        validation="Entire curves marked validation are excluded from optimization.",
        input_sources=sources,
        versions={"python": platform.python_version(), "numpy": np.__version__,
                  "scipy": scipy.__version__, "pandas": pd.__version__,
                  "matplotlib": matplotlib.__version__})
    write_json(output_dir / "run_configuration.json", config)
    parameters, metric_tables, sample_tables, all_runs = [], [], [], []
    with PdfPages(output_dir / "ACM2_extraction_curves.pdf") as pdf:
        for d in devices:
            best, runs = fit_device(d, output_dir, profile, seeds, bounds, weights, current_floor)
            parameters.append(dict(device_id=d.device_id, polarity=d.polarity, card=model_name(d),
                W_um=d.W_um, L_um=d.L_um, temperature_C=d.temperature_C, m=d.m, **best["parameters"],
                objective=best["objective"], seed=best["seed"], bound_flags=";".join(best["bounds_near"])))
            all_runs += [dict(device_id=d.device_id, **r) for r in runs]
            metrics, samples = evaluate(d, best["parameters"], current_floor)
            metric_tables.append(metrics);sample_tables.append(samples)
            folder = output_dir / d.device_id
            metrics.to_csv(folder / "metrics.csv", index=False)
            samples.to_csv(folder / "curves.csv", index=False)
            (folder / f"{model_name(d)}.lib").write_text(card_text(d, best["parameters"]), encoding="utf-8")
            fig = comparison_figure(d, best["parameters"])
            fig.savefig(folder / "comparison.pdf");fig.savefig(folder / "comparison.png", dpi=170)
            pdf.savefig(fig);plt.close(fig)
            pd.DataFrame(parameters).to_csv(output_dir / "parameters.csv", index=False)
    (output_dir / "acm2_cards.lib").write_text("\n".join(
        (output_dir / p["device_id"] / f'{p["card"]}.lib').read_text(encoding="utf-8") for p in parameters), encoding="utf-8")
    metrics = pd.concat(metric_tables, ignore_index=True)
    metrics.to_csv(output_dir / "metrics_all_curves.csv", index=False)
    pd.concat(sample_tables, ignore_index=True).to_csv(output_dir / "curves_all_devices.csv", index=False)
    write_json(output_dir / "all_optimizer_runs.json", all_runs)
    # Include executable code and the exact VA snapshot in each result archive.
    for name in ["acm2_dc.py", "acm2_extraction.py", "acm2.va"]:
        origin = Path(__file__).parent / name
        if not origin.exists():
            origin = Path(__file__).parent / "models" / name
        if origin.exists():
            (output_dir / name).write_bytes(origin.read_bytes())
    files = sorted(p for p in output_dir.rglob("*") if p.is_file())
    write_json(output_dir / "manifest.json", {
        "files": [{"path": p.relative_to(output_dir).as_posix(), "sha256": sha256(p.read_bytes())}
                  for p in files]})
    archive = output_dir.with_suffix(".zip")
    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
        for p in output_dir.rglob("*"):
            if p.is_file(): z.write(p, p.relative_to(output_dir).as_posix())
    return pd.DataFrame(parameters), metrics, archive

In [ ]:
%%writefile acm2_colab_work/acm2.va
//==============================================================================
// ACM2 MOSFET Compact Model - Version 2.0.0
// Advanced Compact MOSFET model, second generation: one module (acm2_va) for
// NMOS and PMOS. Parameters, IHP SG13G2 cards and examples:
// https://github.com/ACMmodel/ACM-MOSFET-models
//
// type = +1 : NMOS   |   type = -1 : PMOS
//
// Copyright (c) 2026, Universidade Federal de Santa Catarina (UFSC)
// Licensed under the Educational Community License, Version 2.0 (ECL-2.0)
//
// Developers:  Deni Germano Alves Neto (Doctoral student, UFSC)
//
// Past developers:
//              Cristina Missel Adornes (UFSC)
//              Gabriel Maranhao (UFSC)
//
// Supervisors: Prof. Carlos Galup-Montoro
//              Prof. Marcio Cherem Schneider
//
//==============================================================================

`include "constants.vams"
`include "disciplines.vams"

// Operating point output macro (CMC-compatible, no external header required)
`define OPP(nam,uni,des) (*units=uni, ask="yes", desc=des*) real nam;
// OPM: quantity of the m parallel devices. m is this module's own instance
// parameter, not the simulator's $mfactor, so $mfactor stays 1 and the
// attribute adds no second scaling.
`define OPM(nam,uni,des) (*units=uni, multiplicity="multiply", ask="yes", desc=des*) real nam;

// Algorithm 443 for normalised charge densities
`define algo_443(Z,qn) \
    if (Z < 0.7385) begin \
        numeratorD   = Z + (4.0/3.0)*Z*Z; \
        denominatorD = 1.0 + (7.0/3.0)*Z + (5.0/6.0)*Z*Z; \
        WnD = numeratorD/denominatorD; \
    end else begin \
        numeratorD   = ln(Z)*ln(Z) + 2.0*ln(Z) - 3.0; \
        denominatorD = 7.0*ln(Z)*ln(Z) + 58.0*ln(Z) + 127.0; \
        WnD = ln(Z) - 24.0*(numeratorD/denominatorD); \
    end \
    ZnD   = ln(Z) - WnD - ln(WnD); \
    TermD = ZnD/(1.0 + WnD); \
    numeratorED   = 2.0*(1.0+WnD)*(1.0+WnD+(2.0/3.0)*ZnD) - ZnD; \
    denominatorED = 2.0*(1.0+WnD)*(1.0+WnD+(2.0/3.0)*ZnD) - 2.0*ZnD; \
    EnD = TermD*(numeratorED/denominatorED); \
    qn  = WnD*(1.0 + EnD); \

module acm2_va(d, g, s, b);

inout  d, g, s, b;
electrical d, g, s, b;
electrical gi;

branch(d,b) BDB;
branch(s,b) BSB;
branch(d,s) BDS;

//----------------------------------------------------------------------
// Parameters
//----------------------------------------------------------------------

// Device type
(*desc="Device type: +1=NMOS, -1=PMOS"*)
parameter integer type = 1 from [-1:1] exclude 0;

// Device dimensions
(*desc="Channel width",                     units="m",    type="instance"*) parameter real w  = 1e-6   from (0:inf);
(*desc="Channel length",                    units="m",    type="instance"*) parameter real l  = 1e-6   from (0:inf);
(*desc="Drain diffusion area",              units="m^2",  type="instance"*) parameter real ad = 2e-12  from [0:inf);
(*desc="Source diffusion area",             units="m^2",  type="instance"*) parameter real as = 2e-12  from [0:inf);
(*desc="Drain diffusion perimeter",         units="m",    type="instance"*) parameter real pd = 10e-6  from [0:inf);
(*desc="Source diffusion perimeter",        units="m",    type="instance"*) parameter real ps = 10e-6  from [0:inf);
(*desc="Number of transistors in parallel", units="",     type="instance"*) parameter real m  = 1      from [1:inf);

// DC parameters
(*desc="Threshold voltage", units="V"*) parameter real vt0   = 0.5  from [0:inf);
(*desc="Sheet specific current I_SH: specific current of one device = ish*w/l", units="A"*) parameter real ish = 5e-6 from (0:inf);
(*desc="Charge-length offset: lc = l - dlc in charges, capacitances, mobility and noise", units="m"*) parameter real dlc = 0.0 from [0:inf);
(*desc="Slope factor"*) parameter real n     = 1.3  from [1:3];
(*desc="Sigma"*) parameter real sigma = 0.03 from [0:inf);
(*desc="Zeta (0: no velocity saturation)"*) parameter real zeta = 0.05 from [0:inf);

// Junction capacitance parameters
(*desc="Zero-bias bottom junction cap",         units="F/m^2"*) parameter real cj0        = 4e-3  from [0:inf);
(*desc="Zero-bias side-wall junction cap",      units="F/m^2"*) parameter real cj0sw      = 4e-3  from [0:inf);
(*desc="Bottom junction grading coeff"*) parameter real xd_mj      = 0.5   from [0:inf);
(*desc="Side-wall junction grading coeff"*) parameter real xd_mjsw    = 0.5   from [0:inf);
(*desc="Bottom junction built-in potential",    units="V"*) parameter real phi_zero   = 0.6   from (0:inf);
(*desc="Side-wall junction built-in potential", units="V"*) parameter real phi_zerosw = 0.6   from (0:inf);

// Process parameters
(*desc="Junction depth",      units="m"*) parameter real xj  = 150e-9 from [0:inf);
(*desc="Gate overlap length",  units="m"*) parameter real ld  = 30e-9  from [0:inf);
(*desc="Oxide thickness",      units="m"*) parameter real tox = 4e-9   from (0:inf);
(*desc="Oxide relative permittivity"*) parameter real epsrox = 3.97 from (0:inf);

// Mismatch parameters
(*desc="vt0 mismatch coefficient", units="V*m", type="instance"*) parameter real avt0 = 0.0 from (-inf:inf);
(*desc="IS mismatch coefficient",  units="m",   type="instance"*) parameter real ak   = 0.0 from [0:inf);

// Temperature parameters
(*desc="Parameter reference temperature", units="K"*)                   parameter real tref       = 300.15  from (0:inf);
(*desc="Device temperature offset",       units="K",   type="instance"*) parameter real dtemp      = 0.0     from (-inf:inf);
aliasparam trise = dtemp;
(*desc="vt0 temperature coefficient",     units="V/K"*) parameter real alphavt0   = -0.4e-3 from [-1:1];
(*desc="IS temperature coefficient"*) parameter real alphais    = 1.5     from [0:inf);
(*desc="Sigma temperature coefficient"*) parameter real alphasigma = 0.3e-6  from [0:inf);
(*desc="Zeta temperature coefficient"*) parameter real alphazeta  = 0.2e-3  from [0:inf);

// Noise parameters
(*desc="Equivalent oxide trap density", units="m^-2"*) parameter real n_ot = 1e12 from [0:inf);
(*desc="Gate resistance per device (divided by m); 0 removes the gate node", units="Ohm", type="instance"*) parameter real rg = 10 from [0:inf);

//----------------------------------------------------------------------
// Internal variables
//----------------------------------------------------------------------
real VP, PhiT, VDB_eff, VSB_eff, VGB_eff, VGD_eff, VGS_eff, VDS_eff;
real qsat, qidsat, q1, q2, dqs;
real X, Y, numeratorD, denominatorD, TermD, numeratorED, denominatorED, ZnD, EnD, WnD;
real id, ID, e0, eox, epsi, We, lc;
real umob, l_vsat, Cox, kboltz, q_e;
real gm_frac1, gm_frac2, gm_frac3, gms, gmd, gm;
real alpha, alpha_frac, alpha_frac2, alpha_frac3;
real QI, QD, QS, QG, QB, QID, QIS, QBwi;
real Cgso, Cgdo, Cdso, Csdo;
real Cgs, Cgd, Cgb, Cbs, Cbd, Cds, Csd, Cm;
real Cgs_ov, Cgd_ov, Cjdb, Cjsb, Cjdb_sw, Cjsb_sw;
real qovd, qovs, qjd, qjs;
real vt0_T, is_T, sigma_T, zeta_T, Tin, isg;
real func_thermal, func_flicker;
real gmin;

//----------------------------------------------------------------------
// Operating point outputs
// The simulator uses the terminal charges (ddt(Q) below). gm/gms/gmd and the
// intrinsic C*_op are analytical estimates, not exact derivatives of the
// simulated current and charges; use a native AC analysis for that response.
// Cjsb_op/Cjdb_op are exact: the junction charges are integrals of C(V).
//----------------------------------------------------------------------
`OPP(qS_op,     "",          "Normalised source charge density")
`OPP(qD_op,     "",          "Normalised drain charge density")
`OPM(Cgs_op,    "F",         "Gate-source capacitance (analytical estimate)")
`OPM(Cgd_op,    "F",         "Gate-drain capacitance (analytical estimate)")
`OPM(Cgb_op,    "F",         "Gate-bulk capacitance (analytical estimate)")
`OPM(Cds_op,    "F",         "Drain-source capacitance (analytical estimate)")
`OPM(Csd_op,    "F",         "Source-drain capacitance (analytical estimate)")
`OPM(Cbs_op,    "F",         "Bulk-source capacitance (analytical estimate)")
`OPM(Cbd_op,    "F",         "Bulk-drain capacitance (analytical estimate)")
`OPM(Cm_op,     "F",         "Nonreciprocal coefficient (Csd-Cds)/n (analytical estimate)")
`OPM(Cgs_ov_op, "F",         "Gate-source overlap capacitance")
`OPM(Cgd_ov_op, "F",         "Gate-drain overlap capacitance")
`OPM(Cjsb_op,   "F",         "Source-bulk junction capacitance")
`OPM(Cjdb_op,   "F",         "Drain-bulk junction capacitance")
`OPM(gm_op,     "A/V",       "Gate transconductance (analytical estimate)")
`OPM(gms_op,    "A/V",       "Source transconductance (analytical estimate)")
`OPM(gmd_op,    "A/V",       "Drain transconductance (analytical estimate)")
`OPP(umob_op,   "m^2/(V*s)", "Carrier mobility")

//----------------------------------------------------------------------
// Junction depletion charge: integral of C(v) from 0 to V (V > 0 reverse bias),
// C(v) = c0*(1 + v/phi)^(-mj) for v > 0 and c0*(1 - mj*v/phi) for v <= 0,
// so that dQ/dV = C(V) on both sides and Q, C and dC/dV are continuous at 0.
//----------------------------------------------------------------------
analog function real qdep;
    input v, c0, phi, mj;
    real v, c0, phi, mj;
    begin
        if (v > 0.0) begin
            if (abs(1.0 - mj) < 1e-6)
                qdep = c0*phi*ln(1.0 + v/phi);
            else
                qdep = c0*phi*(exp((1.0 - mj)*ln(1.0 + v/phi)) - 1.0)/(1.0 - mj);
        end else
            qdep = c0*(v - 0.5*mj*v*v/phi);
    end
endfunction

analog begin

    //------------------------------------------------------------------
    // Constants and geometry
    //------------------------------------------------------------------
    e0     = `P_EPS0;
    eox    = epsrox * e0;
    q_e    = `P_Q;
    kboltz = `P_K;
    epsi   = 0.1;
    gmin   = $simparam("gmin", 0.0);
    We     = m * w;
    // Charge length; dlc < l is required (lc > 0).
    lc     = l - dlc;
    PhiT   = $vt($temperature + dtemp);

    //------------------------------------------------------------------
    // Specific current of one device: I_S = I_SH*W/L
    //------------------------------------------------------------------
    isg = ish*w/l;

    //------------------------------------------------------------------
    // Temperature scaling
    //------------------------------------------------------------------
    Tin     = $temperature + dtemp;
    vt0_T   = vt0  + alphavt0  *(Tin - tref) + avt0/sqrt(We*l);
    is_T    = isg  *(1.0 + ((2.0 - alphais)/Tin)*(Tin - tref)) + (isg*ak)/sqrt(We*l);
    sigma_T = sigma + alphasigma*(Tin - tref);
    // zeta_T >= 0; zeta_T = 0 means no velocity saturation.
    zeta_T  = max(zeta + alphazeta*(Tin - tref), 0.0);

    //------------------------------------------------------------------
    // Effective voltages - type-scaled so all physics stays NMOS-like
    //------------------------------------------------------------------
    VGB_eff = type * V(gi,b);
    VDB_eff = type * V(BDB);
    VSB_eff = type * V(BSB);
    VDS_eff = type * V(BDS);
    VGD_eff = VGB_eff - VDB_eff;
    VGS_eff = VGB_eff - VSB_eff;

    //------------------------------------------------------------------
    // Internal process parameters
    //------------------------------------------------------------------
    Cox  = eox / tox;
    // Mobility from the single-device width w: is_T is per device, so umob
    // and vsat do not depend on m.
    umob   = (is_T * lc) / (Cox * n * PhiT * PhiT * w * 0.5);
    // lc/vsat, with vsat = umob*PhiT/(lc*zeta_T); zero when zeta_T = 0.
    l_vsat = (lc * lc * zeta_T) / (umob * PhiT);

    //------------------------------------------------------------------
    // Pinch-off voltage (ACM2 core)
    //------------------------------------------------------------------
    VP = (VGB_eff - vt0_T + sigma_T*(VDB_eff + VSB_eff)) / n;

    //------------------------------------------------------------------
    // Normalised charge densities (Algorithm 443 + UCCM)
    //------------------------------------------------------------------
    X = exp(((VP - VSB_eff)/PhiT) + 1.0);
    `algo_443(X, q1)

    // q1 - qsat in cancellation-free form (valid for zeta_T = 0).
    dqs  = 2.0*q1/(1.0 + zeta_T + sqrt((1.0 + zeta_T)*(1.0 + zeta_T) + 2.0*zeta_T*q1));
    qsat = q1 - dqs;

    Y = dqs*exp(-VDS_eff/PhiT)*exp(dqs);
    `algo_443(Y, q2)
    q2 = q2 + qsat;

    //------------------------------------------------------------------
    // Drain current
    //------------------------------------------------------------------
    id = ((q2 + q1 + 2.0)/(1.0 + sqrt(zeta_T*(q1-q2)*zeta_T*(q1-q2) + epsi*epsi)))*(q1 - q2);
    ID = m * is_T * id;
    I(d,s) <+ type * ID;

    //------------------------------------------------------------------
    // Transconductances
    //------------------------------------------------------------------
    // gm, gms, gmd of the m devices, like ID.
    qidsat   = (zeta_T/2.0)*id;
    gm_frac1 = (2.0*m*is_T)/(PhiT*(1.0 + zeta_T*(q1-q2))*(1.0 + zeta_T*(q1-q2)));
    gm_frac2 = (1.0 + q1) + (zeta_T/2.0)*((q1-q2)*(q1-q2));
    gm_frac3 = (1.0 + q2) - (zeta_T/2.0)*((q1-q2)*(q1-q2));
    gms = gm_frac1*( gm_frac2*(1.0-(sigma_T/n))*((q1-qidsat)/(1.0+q1-qidsat))
                   + gm_frac3*(sigma_T/n)        *((q2-qidsat)/(1.0+q2-qidsat)));
    gmd = gm_frac1*( gm_frac2*(sigma_T/n)        *((q1-qidsat)/(1.0+q1-qidsat))
                   + gm_frac3*(1.0-(sigma_T/n))  *((q2-qidsat)/(1.0+q2-qidsat)));
    gm  = (gm_frac1/n)*( gm_frac2*((q1-qidsat)/(1.0+q1-qidsat))
                        - gm_frac3*((q2-qidsat)/(1.0+q2-qidsat)));

    //------------------------------------------------------------------
    // linearity factor
    //------------------------------------------------------------------
    alpha = (1.0 + q2 - qidsat)/(1.0 + q1 - qidsat);

    //------------------------------------------------------------------
    // Total charges
    //------------------------------------------------------------------
    QI    = -n*PhiT*Cox*We*lc*((2.0/3.0)*(((1.0+alpha+alpha*alpha)/(1.0+alpha))
            *(q1+1.0-qidsat)) - 1.0) - l_vsat*ID;
    // Constant-n bulk reference; vt0_T fixes the charge origin at fixed T.
    QBwi = -((n-1.0)/n)*Cox*We*lc*(VGB_eff - vt0_T);
    QB    = -((n-1.0)/n)*QI + QBwi;
    QG    = -QB - QI;
    QD    = -n*PhiT*Cox*We*lc*((2.0/15.0)*(q1+1.0-qidsat)
            *((2.0+4.0*alpha+6.0*alpha*alpha+3.0*alpha*alpha*alpha)/((1.0+alpha)*(1.0+alpha)))
            - 0.5) - 0.5*l_vsat*ID;
    QS    = QI - QD;
    QID   = q2*(-n*Cox*PhiT);
    QIS   = q1*(-n*Cox*PhiT);

    //------------------------------------------------------------------
    // Intrinsic capacitance matrix
    //------------------------------------------------------------------
    Cgso = (2.0/3.0)*We*lc*Cox*((1.0+2.0*alpha)*(q1-qidsat))
                             /((1.0+alpha)*(1.0+alpha)*(1.0+(q1-qidsat)));
    Cgdo = (2.0/3.0)*We*lc*Cox*(((alpha*alpha)+2.0*alpha)*(q2-qidsat))
                             /((1.0+alpha)*(1.0+alpha)*(1.0+(q2-qidsat)));
    Cdso = (-4.0/15.0)*n*We*lc*Cox
           *(((1.0+3.0*alpha+alpha*alpha)*(q1-qidsat))
           /((1.0+alpha)*(1.0+alpha)*(1.0+alpha)*(1.0+(q1-qidsat))));
    Csdo = (-4.0/15.0)*n*We*lc*Cox
           *(((alpha+3.0*alpha*alpha+alpha*alpha*alpha)*(q2-qidsat))
           /((1.0+alpha)*(1.0+alpha)*(1.0+alpha)*(1.0+(q2-qidsat))));

    alpha_frac  = ((1.0-alpha)*(1.0-alpha))/((1.0+alpha)*(1.0+alpha));
    alpha_frac2 = ((3.0*alpha+7.0)*(1.0-alpha)*(1.0-alpha))/((1.0+alpha)*(1.0+alpha)*(1.0+alpha));
    alpha_frac3 = ((3.0+7.0*alpha)*(1.0-alpha)*(1.0-alpha))/((1.0+alpha)*(1.0+alpha)*(1.0+alpha));
    Cgs = Cgso*(1.0-(sigma_T/n)) - Cgdo*(sigma_T/n) + ((l_vsat*gms)/(3.0*n))*alpha_frac;
    Cgd = Cgdo*(1.0-(sigma_T/n)) - Cgso*(sigma_T/n) - ((l_vsat*gmd)/(3.0*n))*alpha_frac;
    Cgb = ((n-1.0)/n)*(We*lc*Cox - Cgs - Cgd);
    Cbs = (n-1.0)*Cgs;
    Cbd = (n-1.0)*Cgd;
    Cds = Cdso*(1.0-(sigma_T/n)) - ((l_vsat*gms)/30.0)*alpha_frac2;
    Csd = Csdo*(1.0-(sigma_T/n)) + ((l_vsat*gmd)/30.0)*alpha_frac3;
    Cm  = (Csd - Cds)/n;

    //------------------------------------------------------------------
    // Extrinsic capacitances
    //------------------------------------------------------------------
    Cgs_ov = Cox*We*ld;
    Cgd_ov = Cox*We*ld;

    if (VDB_eff > 0.0) begin
        Cjdb    = cj0  *ad    *exp(-xd_mj  *ln(1.0 + VDB_eff/phi_zero  ));
        Cjdb_sw = cj0sw*xj*pd *exp(-xd_mjsw*ln(1.0 + VDB_eff/phi_zerosw));
    end else begin
        Cjdb    = cj0  *ad    *(1.0 - xd_mj  *VDB_eff/phi_zero  );
        Cjdb_sw = cj0sw*xj*pd *(1.0 - xd_mjsw*VDB_eff/phi_zerosw);
    end

    if (VSB_eff > 0.0) begin
        Cjsb    = cj0  *as    *exp(-xd_mj  *ln(1.0 + VSB_eff/phi_zero  ));
        Cjsb_sw = cj0sw*xj*ps *exp(-xd_mjsw*ln(1.0 + VSB_eff/phi_zerosw));
    end else begin
        Cjsb    = cj0  *as    *(1.0 - xd_mj  *VSB_eff/phi_zero  );
        Cjsb_sw = cj0sw*xj*ps *(1.0 - xd_mjsw*VSB_eff/phi_zerosw);
    end

    qovd = Cgd_ov * VGD_eff;
    qovs = Cgs_ov * VGS_eff;
    // ad, as, pd, ps are per device, like w: the m devices add their junctions.
    // Charges are integrals of the C(V) above, so dQ/dV = C(V).
    qjd  = m*(qdep(VDB_eff, cj0*ad, phi_zero, xd_mj) + qdep(VDB_eff, cj0sw*xj*pd, phi_zerosw, xd_mjsw));
    qjs  = m*(qdep(VSB_eff, cj0*as, phi_zero, xd_mj) + qdep(VSB_eff, cj0sw*xj*ps, phi_zerosw, xd_mjsw));

    //------------------------------------------------------------------
    // Operating point assignments
    //------------------------------------------------------------------
    begin : OPval
        qS_op     = q1;
        qD_op     = q2;
        Cgs_op    = Cgs;
        Cgd_op    = Cgd;
        Cgb_op    = Cgb;
        Cds_op    = Cds;
        Csd_op    = Csd;
        Cbs_op    = Cbs;
        Cbd_op    = Cbd;
        Cm_op     = Cm;
        Cgs_ov_op = Cgs_ov;
        Cgd_ov_op = Cgd_ov;
        Cjsb_op   = m*(Cjsb + Cjsb_sw);
        Cjdb_op   = m*(Cjdb + Cjdb_sw);
        gm_op     = gm;
        gms_op    = gms;
        gmd_op    = gmd;
        umob_op   = umob;
    end

    //------------------------------------------------------------------
    // Transient contributions
    // gmin uses physical V(d,b)/V(s,b): type*VDB_eff = V(d,b)
    //------------------------------------------------------------------
    I(gi,b) <+ type * ddt(QG);
    I(d,b)  <+ type * ddt(QD) + V(BDB)*gmin;
    I(s,b)  <+ type * ddt(QS) + V(BSB)*gmin;
    I(gi,d) <+ type * ddt(qovd);
    I(gi,s) <+ type * ddt(qovs);
    I(d,b) <+ type * ddt(qjd);
    I(s,b) <+ type * ddt(qjs);

    //------------------------------------------------------------------
    // Noise
    //------------------------------------------------------------------
    // QI is the negative (NMOS-frame) inversion charge: the PSD is -4kT*umob*QI/lc^2.
    func_thermal = -(4.0*kboltz*Tin*umob*QI)/(lc*lc);

    // ln(...) and ID both have the sign of q1 - q2, so the
    // PSD is non-negative in both directions and zero at V_DS = 0.
    func_flicker = (q_e*q_e*n_ot*umob/(lc*lc*n*Cox))
                 * ln((n*Cox*PhiT - QIS)/(n*Cox*PhiT - QID))*ID;
    I(d,s) <+ white_noise(func_thermal, "thermal") + flicker_noise(func_flicker, 1.0, "flicker");

    //------------------------------------------------------------------
    // Gate resistance
    //------------------------------------------------------------------
    // m parallel gates: rg/m. rg = 0 connects g to gi directly.
    if (rg > 0.0) begin
        I(g, gi) <+ V(g,gi)*m/rg;
        I(g, gi) <+ white_noise(4.0*kboltz*Tin*m/rg, "rg_noise");
    end else begin
        V(g, gi) <+ 0.0;
    end

end
endmodule

In [ ]:
import json
ASSETS = {'ihp_examples.json': '{\n "repository": "https://github.com/IHP-GmbH/IHP-Open-PDK",\n "commit": "5e6d592e4002946a4616f798c357f0f3c06cf3b6",\n "license": "Apache License 2.0 (IHP-Open-PDK)",\n "source_kind": "measured DC I-V, ICCAP MDM",\n "body_source_bias": "only VS = VB = 0 blocks are used",\n "temperature_note": "ICCAP headers state 27 C (300.15 K); file names say 300K",\n "polarity_note": "PMOS voltages and currents are negated on loading (type = -1 mapping of the ACM2 code)",\n "devices": [\n  {\n   "device_id": "IHP_nmos_W0.15um_L0.12um",\n   "polarity": "nmos",\n   "W_um": 0.15,\n   "L_um": 0.12,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_nmos~W0u15_L0u12_S546_2~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u15_L0u12_S546_2~dc_idvd_vth~300K.mdm",\n     "sha256": "282611feb3bda98d2f41f7baac9743f49404356169b57ec036e88d5674d7adb3"\n    },\n    {\n     "filename": "SG13_nmos~W0u15_L0u12_S546_2~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u15_L0u12_S546_2~dc_idvd~300K.mdm",\n     "sha256": "31ed19cf039b8e3874b577995ae1270cc1b04846d89d5ed2773b6ea7eb7fb3c9"\n    },\n    {\n     "filename": "SG13_nmos~W0u15_L0u12_S546_2~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u15_L0u12_S546_2~dc_idvg~300K.mdm",\n     "sha256": "e96bbb20420470ffa75ebd4876636d12a06d6e82e9dd8cbb1e97ab0945cd3af3"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_nmos_W10um_L0.12um",\n   "polarity": "nmos",\n   "W_um": 10.0,\n   "L_um": 0.12,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_nmos~W10u0_L0u12_S540_1~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W10u0_L0u12_S540_1~dc_idvd_vth~300K.mdm",\n     "sha256": "4055bbe2ae3cea5cc93c6e8e73c0202b71f2047ee4e15bcf7260e41e22361abf"\n    },\n    {\n     "filename": "SG13_nmos~W10u0_L0u12_S540_1~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W10u0_L0u12_S540_1~dc_idvd~300K.mdm",\n     "sha256": "9b60d79dc09ecd85d6d8c1c9492f45781b2af6986bf850616a207fd311d024c4"\n    },\n    {\n     "filename": "SG13_nmos~W10u0_L0u12_S540_1~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W10u0_L0u12_S540_1~dc_idvg~300K.mdm",\n     "sha256": "7d08ba216183ebccc7961b831acafed849dc1bd82218a8b2e3917c3e1477e7a9"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_nmos_W0.15um_L0.13um",\n   "polarity": "nmos",\n   "W_um": 0.15,\n   "L_um": 0.13,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_nmos~W0u15_L0u13_S546_3~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u15_L0u13_S546_3~dc_idvd_vth~300K.mdm",\n     "sha256": "b1fb5e86a809fa5af9232c45dd55973d3e6a07edcb6adda4dd886965d9d9c8b7"\n    },\n    {\n     "filename": "SG13_nmos~W0u15_L0u13_S546_3~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u15_L0u13_S546_3~dc_idvd~300K.mdm",\n     "sha256": "6a4785b6a8bc41efdba614777ac454127575c45b31ee2131a7f03f9dfa6a4f91"\n    },\n    {\n     "filename": "SG13_nmos~W0u15_L0u13_S546_3~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u15_L0u13_S546_3~dc_idvg~300K.mdm",\n     "sha256": "4a906ec262402ce24d0d5547c655638b4d702768bdcde52635de2371095da86d"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_nmos_W0.18um_L0.13um",\n   "polarity": "nmos",\n   "W_um": 0.18,\n   "L_um": 0.13,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_nmos~W0u18_L0u13_S545_3~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u18_L0u13_S545_3~dc_idvd_vth~300K.mdm",\n     "sha256": "bb530e02fec678586f103127d8285fe99238c679c49d803d5445f883fb17ecaf"\n    },\n    {\n     "filename": "SG13_nmos~W0u18_L0u13_S545_3~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u18_L0u13_S545_3~dc_idvd~300K.mdm",\n     "sha256": "3c85564585f189d1c6df04f12613cd13403f8330908686b7cd56f072fb0451ae"\n    },\n    {\n     "filename": "SG13_nmos~W0u18_L0u13_S545_3~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u18_L0u13_S545_3~dc_idvg~300K.mdm",\n     "sha256": "8aaf492a032ae866f25a805ed214994e929c702bf49677954cd1fc01a7da182d"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_nmos_W0.3um_L0.13um",\n   "polarity": "nmos",\n   "W_um": 0.3,\n   "L_um": 0.13,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_nmos~W0u3_L0u13_S544_2~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u3_L0u13_S544_2~dc_idvd_vth~300K.mdm",\n     "sha256": "092d970f5bcaa0d05deaa39f2920d66c27f555ba07d43913e7a4827eb12a5ba2"\n    },\n    {\n     "filename": "SG13_nmos~W0u3_L0u13_S544_2~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u3_L0u13_S544_2~dc_idvd~300K.mdm",\n     "sha256": "126378c7ecba4a7ee738a9774776ade08936612ba9698f5e90d2580049f6cd8b"\n    },\n    {\n     "filename": "SG13_nmos~W0u3_L0u13_S544_2~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u3_L0u13_S544_2~dc_idvg~300K.mdm",\n     "sha256": "585e2e330e44b01fb79111c75c6f43bc38d6d11da51ca4f22ebbdf310b168e84"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_nmos_W0.6um_L0.13um",\n   "polarity": "nmos",\n   "W_um": 0.6,\n   "L_um": 0.13,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_nmos~W0u6_L0u13_S543_2~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u6_L0u13_S543_2~dc_idvd_vth~300K.mdm",\n     "sha256": "bf9edf177f1d8f2d08611ec3df689bc2bc2257dd8ed46d3b06152512e1e05224"\n    },\n    {\n     "filename": "SG13_nmos~W0u6_L0u13_S543_2~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u6_L0u13_S543_2~dc_idvd~300K.mdm",\n     "sha256": "fac1feaceeeb178c09b6f9936414da80778f414e43ce7929f064c1028cf9f893"\n    },\n    {\n     "filename": "SG13_nmos~W0u6_L0u13_S543_2~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u6_L0u13_S543_2~dc_idvg~300K.mdm",\n     "sha256": "c9878beb361d60ee0f54105559cba1510fe7185bdc81fd4f7b13a32b343149b3"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_nmos_W2um_L0.13um",\n   "polarity": "nmos",\n   "W_um": 2.0,\n   "L_um": 0.13,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_nmos~W02u0_L0u13_S542_4~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W02u0_L0u13_S542_4~dc_idvd_vth~300K.mdm",\n     "sha256": "9bc394f1b5cbf037b28becf2f0cb4622b04b2e94b4e8245f91f5ecf50c4bfa43"\n    },\n    {\n     "filename": "SG13_nmos~W02u0_L0u13_S542_4~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W02u0_L0u13_S542_4~dc_idvd~300K.mdm",\n     "sha256": "2fc22e569a2ac7df337517d14b22e2478de5095cba3ecff54e13f338ec36bcd7"\n    },\n    {\n     "filename": "SG13_nmos~W02u0_L0u13_S542_4~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W02u0_L0u13_S542_4~dc_idvg~300K.mdm",\n     "sha256": "00fda93175dcd811c247cd8b0a90d8885fa0e9c332dab7f0609bd6aa5b4bebe9"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_nmos_W5um_L0.13um",\n   "polarity": "nmos",\n   "W_um": 5.0,\n   "L_um": 0.13,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_nmos~W05u0_L0u13_S542_1~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W05u0_L0u13_S542_1~dc_idvd_vth~300K.mdm",\n     "sha256": "223f6bb180b81c6c94d6c95f5a050cb4d4c5e8aad4b7644293393f186e805075"\n    },\n    {\n     "filename": "SG13_nmos~W05u0_L0u13_S542_1~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W05u0_L0u13_S542_1~dc_idvd~300K.mdm",\n     "sha256": "c7d864e98d768fb7f626bee9976d8dda97955b225495e15f9e061aa6cc79f8fd"\n    },\n    {\n     "filename": "SG13_nmos~W05u0_L0u13_S542_1~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W05u0_L0u13_S542_1~dc_idvg~300K.mdm",\n     "sha256": "08c3df6a8872883e69af7db68aa0756a1b3560972affe0fd0d57ed633a43cb1c"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_nmos_W10um_L0.13um",\n   "polarity": "nmos",\n   "W_um": 10.0,\n   "L_um": 0.13,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_nmos~W10u0_L0u13_S540_2~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W10u0_L0u13_S540_2~dc_idvd_vth~300K.mdm",\n     "sha256": "d8f877c4d1afcc92b11ee94a67727e7c0f7099375667cc86a1fa8e3f160cd999"\n    },\n    {\n     "filename": "SG13_nmos~W10u0_L0u13_S540_2~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W10u0_L0u13_S540_2~dc_idvd~300K.mdm",\n     "sha256": "1e48f43213bfb818baba9e62190ca5aadb12ec527deeea5d374dc09c060a9992"\n    },\n    {\n     "filename": "SG13_nmos~W10u0_L0u13_S540_2~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W10u0_L0u13_S540_2~dc_idvg~300K.mdm",\n     "sha256": "d7b3ead182073e54466e9d402e1197ed31e8701a4058ecf35cb6f4aa83e6f22d"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_nmos_W0.15um_L0.14um",\n   "polarity": "nmos",\n   "W_um": 0.15,\n   "L_um": 0.14,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_nmos~W0u15_L0u14_S546_4~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u15_L0u14_S546_4~dc_idvd_vth~300K.mdm",\n     "sha256": "2dc90455551a62f987dab3660d983a0c275f5f779ad3013b3efb1bc4f0dc8c90"\n    },\n    {\n     "filename": "SG13_nmos~W0u15_L0u14_S546_4~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u15_L0u14_S546_4~dc_idvd~300K.mdm",\n     "sha256": "668d95662190ac162285d359126a01ba94a7b10eb86f4f6b5877717ea7948158"\n    },\n    {\n     "filename": "SG13_nmos~W0u15_L0u14_S546_4~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u15_L0u14_S546_4~dc_idvg~300K.mdm",\n     "sha256": "1272572307de03c73294767f617ce200dc4a300b9bf20a7cd8c38f780598eb52"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_nmos_W0.18um_L0.14um",\n   "polarity": "nmos",\n   "W_um": 0.18,\n   "L_um": 0.14,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_nmos~W0u18_L0u14_S545_4~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u18_L0u14_S545_4~dc_idvd_vth~300K.mdm",\n     "sha256": "a1360d79e90295ba91565bd89870ec6e3011e20b7eadc23d0cdc1c456cc1284b"\n    },\n    {\n     "filename": "SG13_nmos~W0u18_L0u14_S545_4~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u18_L0u14_S545_4~dc_idvd~300K.mdm",\n     "sha256": "617cb124d42d5073de4ebd0f72ee5e30f91d1594d2861102b5de49ed72b57dbc"\n    },\n    {\n     "filename": "SG13_nmos~W0u18_L0u14_S545_4~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u18_L0u14_S545_4~dc_idvg~300K.mdm",\n     "sha256": "702fb98dbaa3c04e3ec5e9b9a9eec94030fe6bbc563732082e3f619d9d92d540"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_nmos_W0.3um_L0.14um",\n   "polarity": "nmos",\n   "W_um": 0.3,\n   "L_um": 0.14,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_nmos~W0u3_L0u14_S544_3~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u3_L0u14_S544_3~dc_idvd_vth~300K.mdm",\n     "sha256": "87f65b3c13f0ac9cd640b30614c1bab4aeb5b15b1c1110c938d8a393f5f148a2"\n    },\n    {\n     "filename": "SG13_nmos~W0u3_L0u14_S544_3~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u3_L0u14_S544_3~dc_idvd~300K.mdm",\n     "sha256": "50166db0739466b6f29346f0a35040be393eb682881202682fecde644236a26d"\n    },\n    {\n     "filename": "SG13_nmos~W0u3_L0u14_S544_3~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u3_L0u14_S544_3~dc_idvg~300K.mdm",\n     "sha256": "4672d0922dc96acaf88572fa9f35b30d244fdd17e2fd5a869204d59e6b976b05"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_nmos_W10um_L0.14um",\n   "polarity": "nmos",\n   "W_um": 10.0,\n   "L_um": 0.14,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_nmos~W10u0_L0u14_S540_3~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W10u0_L0u14_S540_3~dc_idvd_vth~300K.mdm",\n     "sha256": "37a0e12a1cc64427983cdb0b2b8fe1199ed7416a13d9e53bf01fbadfd0eb4e5c"\n    },\n    {\n     "filename": "SG13_nmos~W10u0_L0u14_S540_3~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W10u0_L0u14_S540_3~dc_idvd~300K.mdm",\n     "sha256": "1359e4acff25994e4241328416251e9eb7f39b8a00483823e0eb162c8c253286"\n    },\n    {\n     "filename": "SG13_nmos~W10u0_L0u14_S540_3~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W10u0_L0u14_S540_3~dc_idvg~300K.mdm",\n     "sha256": "2ad87a8695c86629576a8e27ea73c2680886793f7ac62833b0ce7ac71f86f5a5"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_nmos_W0.15um_L0.15um",\n   "polarity": "nmos",\n   "W_um": 0.15,\n   "L_um": 0.15,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_nmos~W0u15_L0u15_S546_5~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u15_L0u15_S546_5~dc_idvd_vth~300K.mdm",\n     "sha256": "894df4c912d40f46cc94a9ae67bf666a1e204d277466ecd48d9a4dc9cb809492"\n    },\n    {\n     "filename": "SG13_nmos~W0u15_L0u15_S546_5~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u15_L0u15_S546_5~dc_idvd~300K.mdm",\n     "sha256": "31ba5c99c8e6cff9ec35c21084a2d25b57fb3f6cb1318856e3a9adbeed818830"\n    },\n    {\n     "filename": "SG13_nmos~W0u15_L0u15_S546_5~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u15_L0u15_S546_5~dc_idvg~300K.mdm",\n     "sha256": "36bce3bdb9b4990f62362a873cf5acb08c83932056d9c6c7111947fb81b0641c"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_nmos_W0.18um_L0.15um",\n   "polarity": "nmos",\n   "W_um": 0.18,\n   "L_um": 0.15,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_nmos~W0u18_L0u15_S545_5~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u18_L0u15_S545_5~dc_idvd_vth~300K.mdm",\n     "sha256": "7ee7a83a420603ed13439b36c364ce06a6ca5873dc3fa63a2033d391cb703f0a"\n    },\n    {\n     "filename": "SG13_nmos~W0u18_L0u15_S545_5~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u18_L0u15_S545_5~dc_idvd~300K.mdm",\n     "sha256": "0765a38d6ce607dfea43f8279a56e0de1c5723d69a08ef64a577e18926e7f475"\n    },\n    {\n     "filename": "SG13_nmos~W0u18_L0u15_S545_5~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u18_L0u15_S545_5~dc_idvg~300K.mdm",\n     "sha256": "847c3c1672b1ba4dbe2c310a31763f26b187ad4a6bb4718dc567911001a81628"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_nmos_W0.3um_L0.15um",\n   "polarity": "nmos",\n   "W_um": 0.3,\n   "L_um": 0.15,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_nmos~W0u3_L0u15_S544_4~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u3_L0u15_S544_4~dc_idvd_vth~300K.mdm",\n     "sha256": "19dd7325ea7b5a1bb5a915ae2f4597b23e5c303b6058da1cf3eb2f92d785aa23"\n    },\n    {\n     "filename": "SG13_nmos~W0u3_L0u15_S544_4~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u3_L0u15_S544_4~dc_idvd~300K.mdm",\n     "sha256": "805f068a76ea967b8ce4c1a3e013e9f896af9fe9418007bf6c6eb4318a5118d7"\n    },\n    {\n     "filename": "SG13_nmos~W0u3_L0u15_S544_4~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u3_L0u15_S544_4~dc_idvg~300K.mdm",\n     "sha256": "406d61654926cd0e4919f4822ac3ce55e7b19923eff6bf7fef7c974297c4ca10"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_nmos_W10um_L0.15um",\n   "polarity": "nmos",\n   "W_um": 10.0,\n   "L_um": 0.15,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_nmos~W10u0_L0u15_S540_4~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W10u0_L0u15_S540_4~dc_idvd_vth~300K.mdm",\n     "sha256": "1054a53d0e80aaf153c99d8bd2f66afd83d7d6d8ed11f4226025c95fbd666a51"\n    },\n    {\n     "filename": "SG13_nmos~W10u0_L0u15_S540_4~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W10u0_L0u15_S540_4~dc_idvd~300K.mdm",\n     "sha256": "5343bddd604fcebd3fd53397125cc1ae1a9b8a4bd54b08be4d9072742a2fa34f"\n    },\n    {\n     "filename": "SG13_nmos~W10u0_L0u15_S540_4~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W10u0_L0u15_S540_4~dc_idvg~300K.mdm",\n     "sha256": "a5a8670d440afac608a08c4434e5550b93688f9224e1e45e0bb147f62cfb1719"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_nmos_W0.15um_L0.18um",\n   "polarity": "nmos",\n   "W_um": 0.15,\n   "L_um": 0.18,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_nmos~W0u15_L0u18_S547_1~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u15_L0u18_S547_1~dc_idvd_vth~300K.mdm",\n     "sha256": "bf6dd39faead14eec3eb0afb7397f89a3eaf3b1127ed9f75d3bd86230a580379"\n    },\n    {\n     "filename": "SG13_nmos~W0u15_L0u18_S547_1~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u15_L0u18_S547_1~dc_idvd~300K.mdm",\n     "sha256": "430f0495a9d61ad96857f150456507c4e51f78a8672ad7e632c1551e4307225e"\n    },\n    {\n     "filename": "SG13_nmos~W0u15_L0u18_S547_1~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u15_L0u18_S547_1~dc_idvg~300K.mdm",\n     "sha256": "aeebfade97e7be961565f4361574a847b4850814e00695f7965d44b2e847df31"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_nmos_W0.3um_L0.18um",\n   "polarity": "nmos",\n   "W_um": 0.3,\n   "L_um": 0.18,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_nmos~W0u3_L0u18_S544_5~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u3_L0u18_S544_5~dc_idvd_vth~300K.mdm",\n     "sha256": "5d5afa376f73fb80e2a44be3902ab3790de3f8ff96af90ed19a6795afb0c075b"\n    },\n    {\n     "filename": "SG13_nmos~W0u3_L0u18_S544_5~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u3_L0u18_S544_5~dc_idvd~300K.mdm",\n     "sha256": "881ce8fc4358dcfcbedc22f85fedef39c04d010ce2782091ea4affdf201e7f2a"\n    },\n    {\n     "filename": "SG13_nmos~W0u3_L0u18_S544_5~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u3_L0u18_S544_5~dc_idvg~300K.mdm",\n     "sha256": "2acd06971694c194370f7a2945678df147a6882d6bf7e4819a2a215956e4299a"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_nmos_W0.6um_L0.18um",\n   "polarity": "nmos",\n   "W_um": 0.6,\n   "L_um": 0.18,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_nmos~W0u6_L0u18_S543_3~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u6_L0u18_S543_3~dc_idvd_vth~300K.mdm",\n     "sha256": "32bf150915967899cbf13a15e0e65abf5f03eb1d3041351205f9f4516f061226"\n    },\n    {\n     "filename": "SG13_nmos~W0u6_L0u18_S543_3~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u6_L0u18_S543_3~dc_idvd~300K.mdm",\n     "sha256": "505b326693597057cc886c9bfc2dc77ba8d4f32d215ebb57c5f119918a7525b0"\n    },\n    {\n     "filename": "SG13_nmos~W0u6_L0u18_S543_3~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u6_L0u18_S543_3~dc_idvg~300K.mdm",\n     "sha256": "b1e63419c3aaece587a8726308f206415783b50bf449216aaefcec0e2b54a8c5"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_nmos_W10um_L0.18um",\n   "polarity": "nmos",\n   "W_um": 10.0,\n   "L_um": 0.18,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_nmos~W10u0_L0u18_S540_5~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W10u0_L0u18_S540_5~dc_idvd_vth~300K.mdm",\n     "sha256": "66217d7883d1d36a90c9f7da58d07757061a62e3df8e30db2b00cde7afdea15b"\n    },\n    {\n     "filename": "SG13_nmos~W10u0_L0u18_S540_5~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W10u0_L0u18_S540_5~dc_idvd~300K.mdm",\n     "sha256": "1ce84ed08c5c406b7e9e98a674e2bb65eb35f031fa3da9aa2540ff3485fd314f"\n    },\n    {\n     "filename": "SG13_nmos~W10u0_L0u18_S540_5~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W10u0_L0u18_S540_5~dc_idvg~300K.mdm",\n     "sha256": "5bac5e763f2aac55539d00abaab239af21dbf73d1e42d44d27645c0db3685055"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_nmos_W0.15um_L0.5um",\n   "polarity": "nmos",\n   "W_um": 0.15,\n   "L_um": 0.5,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_nmos~W0u15_L0u5_S547_2~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u15_L0u5_S547_2~dc_idvd_vth~300K.mdm",\n     "sha256": "ae936f7a495bf217defb6de028acf900f1338808dde24ca4b5819da9bf041f8d"\n    },\n    {\n     "filename": "SG13_nmos~W0u15_L0u5_S547_2~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u15_L0u5_S547_2~dc_idvd~300K.mdm",\n     "sha256": "9420baf8b0a0e3967716234690fa4b04262f853b4c998a8736402aa88bf2a9d1"\n    },\n    {\n     "filename": "SG13_nmos~W0u15_L0u5_S547_2~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u15_L0u5_S547_2~dc_idvg~300K.mdm",\n     "sha256": "ddc28f27371673bc8e9cb6420c20fd9ec046d671c359a49dcd85fff9fee9b18a"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_nmos_W0.3um_L0.5um",\n   "polarity": "nmos",\n   "W_um": 0.3,\n   "L_um": 0.5,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_nmos~W0u3_L0u5_S545_1~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u3_L0u5_S545_1~dc_idvd_vth~300K.mdm",\n     "sha256": "cd0532199dfc3841b9867230ac5476d016b5cf03758126905bf43531f23edbae"\n    },\n    {\n     "filename": "SG13_nmos~W0u3_L0u5_S545_1~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u3_L0u5_S545_1~dc_idvd~300K.mdm",\n     "sha256": "d953fc505818e9873b7a3ac1bfd157b3c8b8bfe4e3fb3d305458581e58c81335"\n    },\n    {\n     "filename": "SG13_nmos~W0u3_L0u5_S545_1~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u3_L0u5_S545_1~dc_idvg~300K.mdm",\n     "sha256": "8dd2dfbfa876a287e2c7efb123d2408c7b7fac6e5822510aacddb0a26dcc8ce8"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_nmos_W0.6um_L0.5um",\n   "polarity": "nmos",\n   "W_um": 0.6,\n   "L_um": 0.5,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_nmos~W0u6_L0u5_S543_4~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u6_L0u5_S543_4~dc_idvd_vth~300K.mdm",\n     "sha256": "4488f527e6186ce2f471561d4bd55f5decd09b46b5a4695821a6fcb1b52235ba"\n    },\n    {\n     "filename": "SG13_nmos~W0u6_L0u5_S543_4~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u6_L0u5_S543_4~dc_idvd~300K.mdm",\n     "sha256": "e2e7591f22910aeb9c18bdb05f0b1f4a2d475ccb7ae3cc85f8be3a8a4948c82c"\n    },\n    {\n     "filename": "SG13_nmos~W0u6_L0u5_S543_4~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u6_L0u5_S543_4~dc_idvg~300K.mdm",\n     "sha256": "61dff346f7c747c964442c73c10f9cf55f34fa9596a3e8948cddf63b05fe410d"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_nmos_W10um_L0.5um",\n   "polarity": "nmos",\n   "W_um": 10.0,\n   "L_um": 0.5,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_nmos~W10u0_L0u5_S541_1~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W10u0_L0u5_S541_1~dc_idvd_vth~300K.mdm",\n     "sha256": "51087a57d626f26eb686e4e86040a5a0d1500c45f995804dd9242e1befc7f819"\n    },\n    {\n     "filename": "SG13_nmos~W10u0_L0u5_S541_1~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W10u0_L0u5_S541_1~dc_idvd~300K.mdm",\n     "sha256": "a150bc303e6fd6f581977d51c33a79986a5b9b3dca71c72840e4018c13e1295b"\n    },\n    {\n     "filename": "SG13_nmos~W10u0_L0u5_S541_1~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W10u0_L0u5_S541_1~dc_idvg~300K.mdm",\n     "sha256": "a6b71eb358b0ed9abe5982c698270b1691b5088e3bba02a09b39e4145e144285"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_nmos_W0.6um_L1.2um",\n   "polarity": "nmos",\n   "W_um": 0.6,\n   "L_um": 1.2,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_nmos~W0u6_L1u2_S543_5~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u6_L1u2_S543_5~dc_idvd_vth~300K.mdm",\n     "sha256": "a6093a64df5db9392125aeb806fb46122f1e46426d59a40fb51f70eba73481ef"\n    },\n    {\n     "filename": "SG13_nmos~W0u6_L1u2_S543_5~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u6_L1u2_S543_5~dc_idvd~300K.mdm",\n     "sha256": "4dcd22b80d4c9bcfffde36eb8080e323176ab391eaf38d3b2cec2d7a4c109680"\n    },\n    {\n     "filename": "SG13_nmos~W0u6_L1u2_S543_5~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u6_L1u2_S543_5~dc_idvg~300K.mdm",\n     "sha256": "0d571fa4ca3070687d00f8ce20c78f05316bcfb7324dc9a2a3223e96779315be"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_nmos_W10um_L1.2um",\n   "polarity": "nmos",\n   "W_um": 10.0,\n   "L_um": 1.2,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_nmos~W10u0_L1u2_S541_2~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W10u0_L1u2_S541_2~dc_idvd_vth~300K.mdm",\n     "sha256": "c3d97495c8ec8d1b70a5fc688e0ec80532a0effdd8fe72ca34de0ed6213d43e9"\n    },\n    {\n     "filename": "SG13_nmos~W10u0_L1u2_S541_2~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W10u0_L1u2_S541_2~dc_idvd~300K.mdm",\n     "sha256": "9961cc4d9b7a978cf7647addb94f803fe95d78984e049d3e92816bf9e55a1c5c"\n    },\n    {\n     "filename": "SG13_nmos~W10u0_L1u2_S541_2~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W10u0_L1u2_S541_2~dc_idvg~300K.mdm",\n     "sha256": "10eb487d9ea14eac69a992dbe9b6dc74e327cf37e04eb73c6f42cf74a45db62c"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_nmos_W2um_L2um",\n   "polarity": "nmos",\n   "W_um": 2.0,\n   "L_um": 2.0,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_nmos~W02u0_L2u0_S542_5~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W02u0_L2u0_S542_5~dc_idvd_vth~300K.mdm",\n     "sha256": "5a7fc86fa523e3b88402de11767c03521b545ebee742a8eb5fa87092159e581f"\n    },\n    {\n     "filename": "SG13_nmos~W02u0_L2u0_S542_5~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W02u0_L2u0_S542_5~dc_idvd~300K.mdm",\n     "sha256": "bc26e27817ce8515bee997fb63ac2296e4c376d3fb302c609a51a42600f0b0ae"\n    },\n    {\n     "filename": "SG13_nmos~W02u0_L2u0_S542_5~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W02u0_L2u0_S542_5~dc_idvg~300K.mdm",\n     "sha256": "df4b60e5569334e76c821acf459196865fdea1b4ddbd512935f6805375d445eb"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_nmos_W10um_L2um",\n   "polarity": "nmos",\n   "W_um": 10.0,\n   "L_um": 2.0,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_nmos~W10u0_L2u0_S541_3~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W10u0_L2u0_S541_3~dc_idvd_vth~300K.mdm",\n     "sha256": "139634c33ad2bb466a4f17402fa33fe8486a2e2382d545db25fecd0ae49abff0"\n    },\n    {\n     "filename": "SG13_nmos~W10u0_L2u0_S541_3~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W10u0_L2u0_S541_3~dc_idvd~300K.mdm",\n     "sha256": "eaf410793a4b17542fc09e869d21b59d0ce6aa43cfc65b636562fc7b479d3d67"\n    },\n    {\n     "filename": "SG13_nmos~W10u0_L2u0_S541_3~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W10u0_L2u0_S541_3~dc_idvg~300K.mdm",\n     "sha256": "84c28e5111eb29c90f2af6e07625d7836f137ba7183e304a935cd12081310365"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_nmos_W5um_L5um",\n   "polarity": "nmos",\n   "W_um": 5.0,\n   "L_um": 5.0,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_nmos~W05u0_L5u0_S542_2~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W05u0_L5u0_S542_2~dc_idvd_vth~300K.mdm",\n     "sha256": "675873096231a52c9db6c892e8003115df4a4ac9a2ce4c42fa35af8427d2cae3"\n    },\n    {\n     "filename": "SG13_nmos~W05u0_L5u0_S542_2~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W05u0_L5u0_S542_2~dc_idvd~300K.mdm",\n     "sha256": "c24686736ee38bf12a621cfc0c825641e0cb0217c5e0d4c4b369099912a3a963"\n    },\n    {\n     "filename": "SG13_nmos~W05u0_L5u0_S542_2~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W05u0_L5u0_S542_2~dc_idvg~300K.mdm",\n     "sha256": "f5cf94870f444810b84ccba2fdba40c4fda1456911925623bb0f45bad71cbc6a"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_nmos_W10um_L5um",\n   "polarity": "nmos",\n   "W_um": 10.0,\n   "L_um": 5.0,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_nmos~W10u0_L5u0_S541_4~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W10u0_L5u0_S541_4~dc_idvd_vth~300K.mdm",\n     "sha256": "5759e759ae802901e2e16f5b530b540ab804f40c023132c95a2f81158a985c07"\n    },\n    {\n     "filename": "SG13_nmos~W10u0_L5u0_S541_4~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W10u0_L5u0_S541_4~dc_idvd~300K.mdm",\n     "sha256": "963d776a36bc246e350de14f9318d50e43fcd0bb5f3378f988a04586e6474442"\n    },\n    {\n     "filename": "SG13_nmos~W10u0_L5u0_S541_4~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W10u0_L5u0_S541_4~dc_idvg~300K.mdm",\n     "sha256": "cd1e0150ef717d220582ddcece16fdc4dbfd5419665ea2ffa9a03ebb599c99d9"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_nmos_W0.15um_L10um",\n   "polarity": "nmos",\n   "W_um": 0.15,\n   "L_um": 10.0,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_nmos~W0u15_L10u0_S547_3~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u15_L10u0_S547_3~dc_idvd_vth~300K.mdm",\n     "sha256": "008e7fca6b946377ae1ed2a4b754e6019cfc5e375d6c731a6640f34a352bc20f"\n    },\n    {\n     "filename": "SG13_nmos~W0u15_L10u0_S547_3~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u15_L10u0_S547_3~dc_idvd~300K.mdm",\n     "sha256": "e5b036b257d4768ceee916c3260c5b8fe94cb6764d2e4e29463931cc55852b6b"\n    },\n    {\n     "filename": "SG13_nmos~W0u15_L10u0_S547_3~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u15_L10u0_S547_3~dc_idvg~300K.mdm",\n     "sha256": "b1f4867221f31c412273bf3e3e03977541efb46e6b3c68d76427d7aa4612dbbd"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_nmos_W0.18um_L10um",\n   "polarity": "nmos",\n   "W_um": 0.18,\n   "L_um": 10.0,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_nmos~W0u18_L10u0_S546_1~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u18_L10u0_S546_1~dc_idvd_vth~300K.mdm",\n     "sha256": "201e89e59a29c5a14d6d63b40f07d1b222057f535690213b66078fd38930bbc3"\n    },\n    {\n     "filename": "SG13_nmos~W0u18_L10u0_S546_1~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u18_L10u0_S546_1~dc_idvd~300K.mdm",\n     "sha256": "3f4a013c2cdee573e909bee989cb217a8c6daaa7c3ebe71826891c6817b42646"\n    },\n    {\n     "filename": "SG13_nmos~W0u18_L10u0_S546_1~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u18_L10u0_S546_1~dc_idvg~300K.mdm",\n     "sha256": "7b715144a5f130eb8b97e2937fe9d9faa22c1d87c8a5d01ea5c0d98efcc034cb"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_nmos_W0.3um_L10um",\n   "polarity": "nmos",\n   "W_um": 0.3,\n   "L_um": 10.0,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_nmos~W0u3_L10u0_S545_2~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u3_L10u0_S545_2~dc_idvd_vth~300K.mdm",\n     "sha256": "0ffe36b5a5f458f48f4e035a2d1c9b3ddbc35bf72fcee15f9f74c1d8309a1b89"\n    },\n    {\n     "filename": "SG13_nmos~W0u3_L10u0_S545_2~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u3_L10u0_S545_2~dc_idvd~300K.mdm",\n     "sha256": "75946ef0ac90be5b81603489944234d22856190b2792301309702e523fcab00b"\n    },\n    {\n     "filename": "SG13_nmos~W0u3_L10u0_S545_2~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u3_L10u0_S545_2~dc_idvg~300K.mdm",\n     "sha256": "51843e1d7a7f401801a3907cd8f60116521a79c941a51df16f7c4a985a44a75e"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_nmos_W0.6um_L10um",\n   "polarity": "nmos",\n   "W_um": 0.6,\n   "L_um": 10.0,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_nmos~W0u6_L10u0_S544_1~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u6_L10u0_S544_1~dc_idvd_vth~300K.mdm",\n     "sha256": "79545addc12e85104002e708be92ab68d062bb10e62dfcf137c95344e439287b"\n    },\n    {\n     "filename": "SG13_nmos~W0u6_L10u0_S544_1~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u6_L10u0_S544_1~dc_idvd~300K.mdm",\n     "sha256": "1e157f7b802c95a44468daa444303bfef6c1668fdd7163f05861504f0dc08406"\n    },\n    {\n     "filename": "SG13_nmos~W0u6_L10u0_S544_1~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W0u6_L10u0_S544_1~dc_idvg~300K.mdm",\n     "sha256": "5aa8c37fd4af4355e5d1331821124b32dabc8daa207443098692e7257f942f3d"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_nmos_W2um_L10um",\n   "polarity": "nmos",\n   "W_um": 2.0,\n   "L_um": 10.0,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_nmos~W02u0_L10u0_S543_1~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W02u0_L10u0_S543_1~dc_idvd_vth~300K.mdm",\n     "sha256": "2b5fb72b70277283113f40b0a1ca01a7ccb2e406a63b90f41eab9a379ea4c766"\n    },\n    {\n     "filename": "SG13_nmos~W02u0_L10u0_S543_1~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W02u0_L10u0_S543_1~dc_idvd~300K.mdm",\n     "sha256": "c40bcd404b3e159718b31ca23dae10027440a5656100f307e9a758dc0e37eae6"\n    },\n    {\n     "filename": "SG13_nmos~W02u0_L10u0_S543_1~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W02u0_L10u0_S543_1~dc_idvg~300K.mdm",\n     "sha256": "dfc3bd8f6ebc38b1998993c83fb9fd572105086616ba5709277fd1f95e505be3"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_nmos_W5um_L10um",\n   "polarity": "nmos",\n   "W_um": 5.0,\n   "L_um": 10.0,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_nmos~W05u0_L10u0_S542_3~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W05u0_L10u0_S542_3~dc_idvd_vth~300K.mdm",\n     "sha256": "4cf4f6e779f01b5e82d03d0e06a59c4bd548df58a34d0a10050fc635dab73598"\n    },\n    {\n     "filename": "SG13_nmos~W05u0_L10u0_S542_3~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W05u0_L10u0_S542_3~dc_idvd~300K.mdm",\n     "sha256": "cd032e663da6f90548ebdb06622bfa861c2ab7b922b8c9a02ec0bc2ad716fbe8"\n    },\n    {\n     "filename": "SG13_nmos~W05u0_L10u0_S542_3~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W05u0_L10u0_S542_3~dc_idvg~300K.mdm",\n     "sha256": "a170588f7e1ed3ae9bf7928c7e864e981ccf515dcb8e6bafa2c69a2fee5e89d6"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_nmos_W10um_L10um",\n   "polarity": "nmos",\n   "W_um": 10.0,\n   "L_um": 10.0,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_nmos~W10u0_L10u0_S541_5~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W10u0_L10u0_S541_5~dc_idvd_vth~300K.mdm",\n     "sha256": "777624ade3fd869334e0e899cf06ac4e58a73bc75fa66e46a9c8667f8381493f"\n    },\n    {\n     "filename": "SG13_nmos~W10u0_L10u0_S541_5~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W10u0_L10u0_S541_5~dc_idvd~300K.mdm",\n     "sha256": "688e7bd1708c78f4d915f0c5dcdceed52f589375c27c686ec14d5637062ad4f9"\n    },\n    {\n     "filename": "SG13_nmos~W10u0_L10u0_S541_5~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3/SG13_nmos~W10u0_L10u0_S541_5~dc_idvg~300K.mdm",\n     "sha256": "7050d9acb65c16a7b8aeadcec6222b921c551c287b0544099ff7ab617f5e5b5e"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_pmos_W0.15um_L0.12um",\n   "polarity": "pmos",\n   "W_um": 0.15,\n   "L_um": 0.12,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_pmos~W0u15_L0u12_S554_2~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u15_L0u12_S554_2~dc_idvd_vth~300K.mdm",\n     "sha256": "41d9f5fcf3c01eee63110ecd4000f2ebaf3a7a4114f0f418c6d51e8a7109ce86"\n    },\n    {\n     "filename": "SG13_pmos~W0u15_L0u12_S554_2~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u15_L0u12_S554_2~dc_idvd~300K.mdm",\n     "sha256": "2b791f5e61049605f7b08dee6e12b01f02d1f0a805c935501e44fa4c50ebb6b6"\n    },\n    {\n     "filename": "SG13_pmos~W0u15_L0u12_S554_2~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u15_L0u12_S554_2~dc_idvg~300K.mdm",\n     "sha256": "5a9e99ab02d4a73bae59ac7879468d7504b4150c56cdd71c9ae98a275030d150"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_pmos_W10um_L0.12um",\n   "polarity": "pmos",\n   "W_um": 10.0,\n   "L_um": 0.12,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_pmos~W10u0_L0u12_S548_1~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W10u0_L0u12_S548_1~dc_idvd_vth~300K.mdm",\n     "sha256": "1dc074ff2b5c14f701ba7d249d93d252103edf662c3d79fb8c5cd652ec4713aa"\n    },\n    {\n     "filename": "SG13_pmos~W10u0_L0u12_S548_1~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W10u0_L0u12_S548_1~dc_idvd~300K.mdm",\n     "sha256": "c5dedc63c0eccb83199ff52fe5ce202e0c1db7114c07a665f3dfc34bcc025056"\n    },\n    {\n     "filename": "SG13_pmos~W10u0_L0u12_S548_1~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W10u0_L0u12_S548_1~dc_idvg~300K.mdm",\n     "sha256": "adcb41fd1b6e170667df0bc6595bee15104f4e3db03aea82d5b3264f7dfadcb5"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_pmos_W0.15um_L0.13um",\n   "polarity": "pmos",\n   "W_um": 0.15,\n   "L_um": 0.13,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_pmos~W0u15_L0u13_S554_3~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u15_L0u13_S554_3~dc_idvd_vth~300K.mdm",\n     "sha256": "966a1b7ba7d75915c0224690d906e0169e8f0ae8f86bc20fd944d6cb04fb5f75"\n    },\n    {\n     "filename": "SG13_pmos~W0u15_L0u13_S554_3~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u15_L0u13_S554_3~dc_idvd~300K.mdm",\n     "sha256": "e21d94779e9c14dfcbc50f7da23984c3fb860d411fb9bc9fedb99c155564a0b2"\n    },\n    {\n     "filename": "SG13_pmos~W0u15_L0u13_S554_3~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u15_L0u13_S554_3~dc_idvg~300K.mdm",\n     "sha256": "afb421367a86b630826a4c90ab5ac046edb8e9d14846bb15495bb63a67e63479"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_pmos_W0.18um_L0.13um",\n   "polarity": "pmos",\n   "W_um": 0.18,\n   "L_um": 0.13,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_pmos~W0u18_L0u13_S553_3~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u18_L0u13_S553_3~dc_idvd_vth~300K.mdm",\n     "sha256": "c4eb3ef5d803d427d61907d1fce9faca01d7996d74accc6d6d0794a773cd881f"\n    },\n    {\n     "filename": "SG13_pmos~W0u18_L0u13_S553_3~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u18_L0u13_S553_3~dc_idvd~300K.mdm",\n     "sha256": "5f53d9f5eb20cd41c35a998d3902d3cb3346c18c5f52750c4c04fbe2872f1418"\n    },\n    {\n     "filename": "SG13_pmos~W0u18_L0u13_S553_3~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u18_L0u13_S553_3~dc_idvg~300K.mdm",\n     "sha256": "feefcdc9bdc0021fe121fc794b3e4e9caf17957d02cffec46c051c48d851bea9"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_pmos_W0.3um_L0.13um",\n   "polarity": "pmos",\n   "W_um": 0.3,\n   "L_um": 0.13,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_pmos~W0u3_L0u13_S552_2~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u3_L0u13_S552_2~dc_idvd_vth~300K.mdm",\n     "sha256": "341a2008e88d3e0b3c882f205c2c1378558261b156797b128ef37c5b0607af96"\n    },\n    {\n     "filename": "SG13_pmos~W0u3_L0u13_S552_2~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u3_L0u13_S552_2~dc_idvd~300K.mdm",\n     "sha256": "679362d68263dc29d3c3e5b6023aaa71e9203e5d0fc28a3fed1ac3dfadb193db"\n    },\n    {\n     "filename": "SG13_pmos~W0u3_L0u13_S552_2~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u3_L0u13_S552_2~dc_idvg~300K.mdm",\n     "sha256": "f5300fab246e7e5a2064bef380980af7900ce574c9690ef3b7746ccd9cc71a20"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_pmos_W0.6um_L0.13um",\n   "polarity": "pmos",\n   "W_um": 0.6,\n   "L_um": 0.13,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_pmos~W0u6_L0u13_S551_2~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u6_L0u13_S551_2~dc_idvd_vth~300K.mdm",\n     "sha256": "0a2439b23e52819a2efdaffb389681c91ce3dcc85d7fa907c21397a43edf7eba"\n    },\n    {\n     "filename": "SG13_pmos~W0u6_L0u13_S551_2~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u6_L0u13_S551_2~dc_idvd~300K.mdm",\n     "sha256": "d98998a789e2e2f0e03e346927cff79d2211312e639eb966b2bdeae05f60e14c"\n    },\n    {\n     "filename": "SG13_pmos~W0u6_L0u13_S551_2~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u6_L0u13_S551_2~dc_idvg~300K.mdm",\n     "sha256": "c56a5fab81db9d5aadf906fb356c5bbcdcab5445725e0dec6f0ec585ea5f6aeb"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_pmos_W2um_L0.13um",\n   "polarity": "pmos",\n   "W_um": 2.0,\n   "L_um": 0.13,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_pmos~W02u0_L0u13_S550_4~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W02u0_L0u13_S550_4~dc_idvd_vth~300K.mdm",\n     "sha256": "dd8b85fafa7e165f7487cd798408aeac69c5ea101fe247b9e4b501876f124d8f"\n    },\n    {\n     "filename": "SG13_pmos~W02u0_L0u13_S550_4~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W02u0_L0u13_S550_4~dc_idvd~300K.mdm",\n     "sha256": "3c8dbabd36ed8895ea2dd5a6fa06094c00494f56fa986cad7aa17cfbb06449b2"\n    },\n    {\n     "filename": "SG13_pmos~W02u0_L0u13_S550_4~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W02u0_L0u13_S550_4~dc_idvg~300K.mdm",\n     "sha256": "976d185311e3aeed9c2d5d3e374dd46dc00dd565d1afc630b0ab1bc88751b2d0"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_pmos_W5um_L0.13um",\n   "polarity": "pmos",\n   "W_um": 5.0,\n   "L_um": 0.13,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_pmos~W05u0_L0u13_S550_1~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W05u0_L0u13_S550_1~dc_idvd_vth~300K.mdm",\n     "sha256": "a656f4211c20020f997798794c3c31c7a4ac47dc375233345982687d07f64da1"\n    },\n    {\n     "filename": "SG13_pmos~W05u0_L0u13_S550_1~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W05u0_L0u13_S550_1~dc_idvd~300K.mdm",\n     "sha256": "7026bbd9509fdc8708a65b1a4f44f2cecbd1770297d144b14a09d37954e8d467"\n    },\n    {\n     "filename": "SG13_pmos~W05u0_L0u13_S550_1~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W05u0_L0u13_S550_1~dc_idvg~300K.mdm",\n     "sha256": "e4a1206d9b5c93ce9449e385268c1985ab7084c8821b94130d52113a18d50ea2"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_pmos_W10um_L0.13um",\n   "polarity": "pmos",\n   "W_um": 10.0,\n   "L_um": 0.13,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_pmos~W10u0_L0u13_S548_2~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W10u0_L0u13_S548_2~dc_idvd_vth~300K.mdm",\n     "sha256": "179a837b7cb153e32aa0523ad6d6ef0495e04f2ee2d16cfa6fda02821afd9fb8"\n    },\n    {\n     "filename": "SG13_pmos~W10u0_L0u13_S548_2~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W10u0_L0u13_S548_2~dc_idvd~300K.mdm",\n     "sha256": "bf53dafebb7fd72042f9919a026dad6a08122de9b9d14e25680c89b8d3892f94"\n    },\n    {\n     "filename": "SG13_pmos~W10u0_L0u13_S548_2~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W10u0_L0u13_S548_2~dc_idvg~300K.mdm",\n     "sha256": "e5244166a79f26b87a2aa6b4bd494d9cd545d7ab84ee6ce7dc5610c71983c40c"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_pmos_W0.15um_L0.14um",\n   "polarity": "pmos",\n   "W_um": 0.15,\n   "L_um": 0.14,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_pmos~W0u15_L0u14_S554_4~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u15_L0u14_S554_4~dc_idvd_vth~300K.mdm",\n     "sha256": "829bd7e7c19bb9ef8c61aecffa815784fe5911f501c52dec89a53534ba5f6490"\n    },\n    {\n     "filename": "SG13_pmos~W0u15_L0u14_S554_4~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u15_L0u14_S554_4~dc_idvd~300K.mdm",\n     "sha256": "66e7993acb1334fa201b772d8483a8b6e9aef600d430608826055818553126f5"\n    },\n    {\n     "filename": "SG13_pmos~W0u15_L0u14_S554_4~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u15_L0u14_S554_4~dc_idvg~300K.mdm",\n     "sha256": "d4d22ed5971d2948d49ffc84e23d23f3f88c9151c7f2bad1ec4052e1a1833458"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_pmos_W0.18um_L0.14um",\n   "polarity": "pmos",\n   "W_um": 0.18,\n   "L_um": 0.14,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_pmos~W0u18_L0u14_S553_4~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u18_L0u14_S553_4~dc_idvd_vth~300K.mdm",\n     "sha256": "61425a47000a1bc637740dfc70dbaa362ae1d75f232d0981aed8b0eab8855a0e"\n    },\n    {\n     "filename": "SG13_pmos~W0u18_L0u14_S553_4~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u18_L0u14_S553_4~dc_idvd~300K.mdm",\n     "sha256": "3cf56ec96ac5f1f7f50b62dfc06670162086695c0340913e11d30c351aa19dcb"\n    },\n    {\n     "filename": "SG13_pmos~W0u18_L0u14_S553_4~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u18_L0u14_S553_4~dc_idvg~300K.mdm",\n     "sha256": "5c4b5f943ee80f6f06a278fed8a3c3e4fd22b49d89b07a6b3a754094d8f30435"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_pmos_W0.3um_L0.14um",\n   "polarity": "pmos",\n   "W_um": 0.3,\n   "L_um": 0.14,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_pmos~W0u3_L0u14_S552_3~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u3_L0u14_S552_3~dc_idvd_vth~300K.mdm",\n     "sha256": "3fdbc29708bcdfb6aaf3322934d716e372c6dda681b603c230e42878c211925d"\n    },\n    {\n     "filename": "SG13_pmos~W0u3_L0u14_S552_3~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u3_L0u14_S552_3~dc_idvd~300K.mdm",\n     "sha256": "a0bb05fa3a859510efe4b02d9978e6cf1595632fee7176fa5b34ce789a635985"\n    },\n    {\n     "filename": "SG13_pmos~W0u3_L0u14_S552_3~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u3_L0u14_S552_3~dc_idvg~300K.mdm",\n     "sha256": "e4905d3452efc3aefb8d999e77391f48ec51e8850bb0f8534a8d003ee487e0d0"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_pmos_W10um_L0.14um",\n   "polarity": "pmos",\n   "W_um": 10.0,\n   "L_um": 0.14,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_pmos~W10u0_L0u14_S548_3~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W10u0_L0u14_S548_3~dc_idvd_vth~300K.mdm",\n     "sha256": "053cdf90f32596a7f37bb76698069c5b50358e17ce900f1fe2f7ca58f0dd7e06"\n    },\n    {\n     "filename": "SG13_pmos~W10u0_L0u14_S548_3~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W10u0_L0u14_S548_3~dc_idvd~300K.mdm",\n     "sha256": "6713489601b9a1e12c08a39f888bcd5064ceab55c5f5eb0b7234e2bff3c3846c"\n    },\n    {\n     "filename": "SG13_pmos~W10u0_L0u14_S548_3~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W10u0_L0u14_S548_3~dc_idvg~300K.mdm",\n     "sha256": "eed369f0e137be846305439c2c6f9a46522dfc612f5bf68db2cec1aebba9706d"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_pmos_W0.15um_L0.15um",\n   "polarity": "pmos",\n   "W_um": 0.15,\n   "L_um": 0.15,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_pmos~W0u15_L0u15_S554_5~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u15_L0u15_S554_5~dc_idvd_vth~300K.mdm",\n     "sha256": "22ca33aa433648fdec8aa23fa6771fcf10f800d29d5f35640df29a216a3c7c6a"\n    },\n    {\n     "filename": "SG13_pmos~W0u15_L0u15_S554_5~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u15_L0u15_S554_5~dc_idvd~300K.mdm",\n     "sha256": "1ebf010c9fa2c1339e6112cd9379050331f643f0200dbdc99193977de049fae9"\n    },\n    {\n     "filename": "SG13_pmos~W0u15_L0u15_S554_5~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u15_L0u15_S554_5~dc_idvg~300K.mdm",\n     "sha256": "005c9df54a58085c504a3af24881e131aeb0adbbff4d57f039bb332295b56028"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_pmos_W0.18um_L0.15um",\n   "polarity": "pmos",\n   "W_um": 0.18,\n   "L_um": 0.15,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_pmos~W0u18_L0u15_S553_5~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u18_L0u15_S553_5~dc_idvd_vth~300K.mdm",\n     "sha256": "7404f9787cd72c52470cd03664a414df97360a44b2547cad7d1c101ec6ebd26b"\n    },\n    {\n     "filename": "SG13_pmos~W0u18_L0u15_S553_5~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u18_L0u15_S553_5~dc_idvd~300K.mdm",\n     "sha256": "544af389c64d06ee158dc00102a5b9db94393edad7d4db0e38f60f9ee91153ca"\n    },\n    {\n     "filename": "SG13_pmos~W0u18_L0u15_S553_5~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u18_L0u15_S553_5~dc_idvg~300K.mdm",\n     "sha256": "8f409f9b8628a651b97d18974dc19e44f97819fc2e56667b80b09fb8a04c3055"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_pmos_W0.3um_L0.15um",\n   "polarity": "pmos",\n   "W_um": 0.3,\n   "L_um": 0.15,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_pmos~W0u3_L0u15_S552_4~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u3_L0u15_S552_4~dc_idvd_vth~300K.mdm",\n     "sha256": "29be096161084f89e65b22d6bc382d9a6ddbd71a7ed2e7f7ef952f9da601d8e9"\n    },\n    {\n     "filename": "SG13_pmos~W0u3_L0u15_S552_4~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u3_L0u15_S552_4~dc_idvd~300K.mdm",\n     "sha256": "01e911ed887dd8ebfe4e103b08dc63eea94f311ac724e38a32fb9cee15b1a90b"\n    },\n    {\n     "filename": "SG13_pmos~W0u3_L0u15_S552_4~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u3_L0u15_S552_4~dc_idvg~300K.mdm",\n     "sha256": "bdc8274b22cb8604601d071d1d9ff55d1826d26bd5c1c20561297f9188c47fc2"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_pmos_W10um_L0.15um",\n   "polarity": "pmos",\n   "W_um": 10.0,\n   "L_um": 0.15,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_pmos~W10u0_L0u15_S548_4~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W10u0_L0u15_S548_4~dc_idvd_vth~300K.mdm",\n     "sha256": "1addf1f5772b9c02a7000a8231897e2bcdc8c3090e10d2250acfd3e66987c418"\n    },\n    {\n     "filename": "SG13_pmos~W10u0_L0u15_S548_4~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W10u0_L0u15_S548_4~dc_idvd~300K.mdm",\n     "sha256": "8859a94bedb4d330b81d32e58f6bd61f4a779f73c21f2ed8769680be728b145c"\n    },\n    {\n     "filename": "SG13_pmos~W10u0_L0u15_S548_4~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W10u0_L0u15_S548_4~dc_idvg~300K.mdm",\n     "sha256": "1153ea87313cb11cfcd4708e0d825f1803aa6986c663a5b40dd089ef2fb827ea"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_pmos_W0.15um_L0.18um",\n   "polarity": "pmos",\n   "W_um": 0.15,\n   "L_um": 0.18,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_pmos~W0u15_L0u18_S555_1~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u15_L0u18_S555_1~dc_idvd_vth~300K.mdm",\n     "sha256": "1c57a0d36aff5f4d5bb9a05b4677dfd68d7bf597de518d839cbe4217b12c5f21"\n    },\n    {\n     "filename": "SG13_pmos~W0u15_L0u18_S555_1~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u15_L0u18_S555_1~dc_idvd~300K.mdm",\n     "sha256": "c33c6a51e5b11e9b04f4748bb266cf716c98662d957fafbbba31ec753562318f"\n    },\n    {\n     "filename": "SG13_pmos~W0u15_L0u18_S555_1~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u15_L0u18_S555_1~dc_idvg~300K.mdm",\n     "sha256": "823175e8725d57919f21cbbfe3f85b9855bc95c28f62cd68fa4529a43895d5cd"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_pmos_W0.3um_L0.18um",\n   "polarity": "pmos",\n   "W_um": 0.3,\n   "L_um": 0.18,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_pmos~W0u3_L0u18_S552_5~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u3_L0u18_S552_5~dc_idvd_vth~300K.mdm",\n     "sha256": "5d3457eb4e633430eb8e4be7aa4198f39770a530c8169735cfd9e295c08cfc96"\n    },\n    {\n     "filename": "SG13_pmos~W0u3_L0u18_S552_5~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u3_L0u18_S552_5~dc_idvd~300K.mdm",\n     "sha256": "998b318e404a7851a5c93ea193a81be16014f3b3943a40eefd3b1aad71e03b36"\n    },\n    {\n     "filename": "SG13_pmos~W0u3_L0u18_S552_5~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u3_L0u18_S552_5~dc_idvg~300K.mdm",\n     "sha256": "40a07e6fea4bf7aa917156c1e157f5f51bb76e1229cb48276cdd13dcd1d304f8"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_pmos_W0.6um_L0.18um",\n   "polarity": "pmos",\n   "W_um": 0.6,\n   "L_um": 0.18,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_pmos~W0u6_L0u18_S551_3~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u6_L0u18_S551_3~dc_idvd_vth~300K.mdm",\n     "sha256": "5c43e39102743dfd40f1ab5d35be32a2d7be30c7b74e6b62cf46bfef448f3152"\n    },\n    {\n     "filename": "SG13_pmos~W0u6_L0u18_S551_3~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u6_L0u18_S551_3~dc_idvd~300K.mdm",\n     "sha256": "8554da20d9d8f4d17a755c0937a77b7155ddbc901214ad8625b624bdf244fdf7"\n    },\n    {\n     "filename": "SG13_pmos~W0u6_L0u18_S551_3~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u6_L0u18_S551_3~dc_idvg~300K.mdm",\n     "sha256": "939fff4fef315f97bb52f9f2bbc9757ee3e9662078871ef19c95ee1d6e3f2786"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_pmos_W10um_L0.18um",\n   "polarity": "pmos",\n   "W_um": 10.0,\n   "L_um": 0.18,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_pmos~W10u0_L0u18_S548_5~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W10u0_L0u18_S548_5~dc_idvd_vth~300K.mdm",\n     "sha256": "634bf90ce0cb80ca601d549072426f0a4e0e83e79c3c254daf56023e829775ad"\n    },\n    {\n     "filename": "SG13_pmos~W10u0_L0u18_S548_5~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W10u0_L0u18_S548_5~dc_idvd~300K.mdm",\n     "sha256": "0e5e1414114f87796b1bc7088126df7327e9e6100fa01a68cf586fba04715f1c"\n    },\n    {\n     "filename": "SG13_pmos~W10u0_L0u18_S548_5~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W10u0_L0u18_S548_5~dc_idvg~300K.mdm",\n     "sha256": "8520652d9d46a182e4e32505187400604fb5dd3ee63a241950b07e4dd1571ae4"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_pmos_W0.15um_L0.5um",\n   "polarity": "pmos",\n   "W_um": 0.15,\n   "L_um": 0.5,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_pmos~W0u15_L0u5_S555_2~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u15_L0u5_S555_2~dc_idvd_vth~300K.mdm",\n     "sha256": "aa94ba0ae55c2ac47ffda2dcea55328dd024dc46fb0e4605410716e28b8ada1b"\n    },\n    {\n     "filename": "SG13_pmos~W0u15_L0u5_S555_2~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u15_L0u5_S555_2~dc_idvd~300K.mdm",\n     "sha256": "17e2bf8d18b04af7b9ff68e422523363bfde0bfb4d2e9b0da7eb837512690cf0"\n    },\n    {\n     "filename": "SG13_pmos~W0u15_L0u5_S555_2~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u15_L0u5_S555_2~dc_idvg~300K.mdm",\n     "sha256": "119f4e7b02822c47a82c9a914164afa719de09b655285c2bdb2473306c4eb95d"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_pmos_W0.3um_L0.5um",\n   "polarity": "pmos",\n   "W_um": 0.3,\n   "L_um": 0.5,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_pmos~W0u3_L0u5_S553_1~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u3_L0u5_S553_1~dc_idvd_vth~300K.mdm",\n     "sha256": "ed3f7553534b8d172ac3d3611dfd67eec57c5329a556b1110666de3bb3f92309"\n    },\n    {\n     "filename": "SG13_pmos~W0u3_L0u5_S553_1~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u3_L0u5_S553_1~dc_idvd~300K.mdm",\n     "sha256": "c7f4da135164c538d9a0a195b4e33c7e2b05a4c6a96e129bc023e01871f9ed6b"\n    },\n    {\n     "filename": "SG13_pmos~W0u3_L0u5_S553_1~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u3_L0u5_S553_1~dc_idvg~300K.mdm",\n     "sha256": "5adb52da88be854329a9be12162924c5c93d1d6a0578f467f94a9e601b411cf6"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_pmos_W0.6um_L0.5um",\n   "polarity": "pmos",\n   "W_um": 0.6,\n   "L_um": 0.5,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_pmos~W0u6_L0u5_S551_4~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u6_L0u5_S551_4~dc_idvd_vth~300K.mdm",\n     "sha256": "744ab269f7628a24188e90a3f2d05a2cc257638a14efbcfed1c009258e550296"\n    },\n    {\n     "filename": "SG13_pmos~W0u6_L0u5_S551_4~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u6_L0u5_S551_4~dc_idvd~300K.mdm",\n     "sha256": "7b6ec1156ccf39bb221cde940ab2ae975e4d1946b5e496151f3eeac30dd2a52f"\n    },\n    {\n     "filename": "SG13_pmos~W0u6_L0u5_S551_4~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u6_L0u5_S551_4~dc_idvg~300K.mdm",\n     "sha256": "f2ed1205385ae3147a7af5fa33033902e5931e9e8f647c1b8ddf9cedbb9b1b38"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_pmos_W10um_L0.5um",\n   "polarity": "pmos",\n   "W_um": 10.0,\n   "L_um": 0.5,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_pmos~W10u0_L0u5_S549_1~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W10u0_L0u5_S549_1~dc_idvd_vth~300K.mdm",\n     "sha256": "866bb78022e15b9d332ac5f04e68d47eaa8414867b9ad4a69b3c0485981729ad"\n    },\n    {\n     "filename": "SG13_pmos~W10u0_L0u5_S549_1~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W10u0_L0u5_S549_1~dc_idvd~300K.mdm",\n     "sha256": "be4cfdd82b854dad0988a80128812c452d30f1ef9e474929cbf2917b0fd8858b"\n    },\n    {\n     "filename": "SG13_pmos~W10u0_L0u5_S549_1~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W10u0_L0u5_S549_1~dc_idvg~300K.mdm",\n     "sha256": "818044577ef23aeb82655970500e881e95ad927d40295f9ac67642c0e75857db"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_pmos_W0.6um_L1.2um",\n   "polarity": "pmos",\n   "W_um": 0.6,\n   "L_um": 1.2,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_pmos~W0u6_L1u2_S551_5~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u6_L1u2_S551_5~dc_idvd_vth~300K.mdm",\n     "sha256": "ea7238f339c9101a94df9dc7de57930159c0424c2746067ac4c70c52a7ed44ee"\n    },\n    {\n     "filename": "SG13_pmos~W0u6_L1u2_S551_5~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u6_L1u2_S551_5~dc_idvd~300K.mdm",\n     "sha256": "a0a78bc21fc0ca994b4ed882adc29b5b63f6d1a0a4e1bb820a512136ceb90c80"\n    },\n    {\n     "filename": "SG13_pmos~W0u6_L1u2_S551_5~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u6_L1u2_S551_5~dc_idvg~300K.mdm",\n     "sha256": "4c97d7495db73a1ef843ac7d5821f363f96f4748b850a0994aed5f4431a427eb"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_pmos_W10um_L1.2um",\n   "polarity": "pmos",\n   "W_um": 10.0,\n   "L_um": 1.2,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_pmos~W10u0_L1u2_S549_2~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W10u0_L1u2_S549_2~dc_idvd_vth~300K.mdm",\n     "sha256": "0adfbee4dfa78b4777b3753963f937a41d661a77338b3baf4ead307cd922f9d8"\n    },\n    {\n     "filename": "SG13_pmos~W10u0_L1u2_S549_2~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W10u0_L1u2_S549_2~dc_idvd~300K.mdm",\n     "sha256": "bc47bc08a1102b4a77d938b69ef95426a09f2dcd6d20d768f8a8ed9d127821d1"\n    },\n    {\n     "filename": "SG13_pmos~W10u0_L1u2_S549_2~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W10u0_L1u2_S549_2~dc_idvg~300K.mdm",\n     "sha256": "f20c6c9eecde8856326c33ed5c3c49fc96d99f4edb03da5c5089e2623d5a8625"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_pmos_W2um_L2um",\n   "polarity": "pmos",\n   "W_um": 2.0,\n   "L_um": 2.0,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_pmos~W02u0_L2u0_S550_5~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W02u0_L2u0_S550_5~dc_idvd_vth~300K.mdm",\n     "sha256": "849fb6c72e35b7ac58d383afda53fb86755594ebf50ce3734a2f90679d768522"\n    },\n    {\n     "filename": "SG13_pmos~W02u0_L2u0_S550_5~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W02u0_L2u0_S550_5~dc_idvd~300K.mdm",\n     "sha256": "cc29a9b4b9b5575e8dfdcebde205acb59037354f6eafe8d040d579ed30af7599"\n    },\n    {\n     "filename": "SG13_pmos~W02u0_L2u0_S550_5~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W02u0_L2u0_S550_5~dc_idvg~300K.mdm",\n     "sha256": "fc69a258aefff5dfb8ea89ce42a3984d892a6b3b8394f79b7b25a9a540b2a244"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_pmos_W10um_L2um",\n   "polarity": "pmos",\n   "W_um": 10.0,\n   "L_um": 2.0,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_pmos~W10u0_L2u0_S549_3~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W10u0_L2u0_S549_3~dc_idvd_vth~300K.mdm",\n     "sha256": "fc2ed6455b84566c500644cf3d6d414b41e755fe2d1316b7ba0e061fcc20eca0"\n    },\n    {\n     "filename": "SG13_pmos~W10u0_L2u0_S549_3~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W10u0_L2u0_S549_3~dc_idvd~300K.mdm",\n     "sha256": "dc95285579748f163b64078803db15cb73454395a2f89d77bb514d265766ce3a"\n    },\n    {\n     "filename": "SG13_pmos~W10u0_L2u0_S549_3~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W10u0_L2u0_S549_3~dc_idvg~300K.mdm",\n     "sha256": "634c3106064985a4055ca7d7024d6f8be9b692b2b23ab70cda9ea09680514edb"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_pmos_W5um_L5um",\n   "polarity": "pmos",\n   "W_um": 5.0,\n   "L_um": 5.0,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_pmos~W05u0_L5u0_S550_2~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W05u0_L5u0_S550_2~dc_idvd_vth~300K.mdm",\n     "sha256": "540f00f31f08cb5f004c00073571d3fcef778b34be391e932a48b7fa7f32c396"\n    },\n    {\n     "filename": "SG13_pmos~W05u0_L5u0_S550_2~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W05u0_L5u0_S550_2~dc_idvd~300K.mdm",\n     "sha256": "d4aa2924e5ec7fc8cf24650019a84a5fcfa0cd4c608d94285afa33648bcad90b"\n    },\n    {\n     "filename": "SG13_pmos~W05u0_L5u0_S550_2~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W05u0_L5u0_S550_2~dc_idvg~300K.mdm",\n     "sha256": "185ed7c8a4e1bd3d50f4b1502060237e08a070ff49d5e5c119dbe13d3fe73021"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_pmos_W10um_L5um",\n   "polarity": "pmos",\n   "W_um": 10.0,\n   "L_um": 5.0,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_pmos~W10u0_L5u0_S549_4~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W10u0_L5u0_S549_4~dc_idvd_vth~300K.mdm",\n     "sha256": "ca43b04adf3a88bcd4517cf73799929c3551d99b2aa9135f480fb43d87762e43"\n    },\n    {\n     "filename": "SG13_pmos~W10u0_L5u0_S549_4~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W10u0_L5u0_S549_4~dc_idvd~300K.mdm",\n     "sha256": "c33f7b1231edb621a149a6e970ba16cdca9db3e4f1e4e78b2d5a954728172473"\n    },\n    {\n     "filename": "SG13_pmos~W10u0_L5u0_S549_4~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W10u0_L5u0_S549_4~dc_idvg~300K.mdm",\n     "sha256": "a44bf10157f916de1092229c1b8e9083072cccb93fdfe15bc66250c6a4090e5f"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_pmos_W0.15um_L10um",\n   "polarity": "pmos",\n   "W_um": 0.15,\n   "L_um": 10.0,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_pmos~W0u15_L10u0_S555_3~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u15_L10u0_S555_3~dc_idvd_vth~300K.mdm",\n     "sha256": "37f1e2d0a246149c6aa9afa90d314457781cd91b36c04755bbc791a9baad0ea3"\n    },\n    {\n     "filename": "SG13_pmos~W0u15_L10u0_S555_3~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u15_L10u0_S555_3~dc_idvd~300K.mdm",\n     "sha256": "3007ae51b0a201499cc3c39ca8a32ef7caa7549d18e4493cd30aa2500a7b526c"\n    },\n    {\n     "filename": "SG13_pmos~W0u15_L10u0_S555_3~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u15_L10u0_S555_3~dc_idvg~300K.mdm",\n     "sha256": "82f7223f63d5d7064c6ab0db2177a9e76cc9d10e81dbac65c0f38a43670acc4a"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_pmos_W0.18um_L10um",\n   "polarity": "pmos",\n   "W_um": 0.18,\n   "L_um": 10.0,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_pmos~W0u18_L10u0_S554_1~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u18_L10u0_S554_1~dc_idvd_vth~300K.mdm",\n     "sha256": "70a7fc2d5ef89aa41072ecabbd52f8ef66e61725b989af4561dad7c98e7abce9"\n    },\n    {\n     "filename": "SG13_pmos~W0u18_L10u0_S554_1~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u18_L10u0_S554_1~dc_idvd~300K.mdm",\n     "sha256": "4b7967149a985206af933400cf61baefa5d0a5a79dbfb58ddacc698500870992"\n    },\n    {\n     "filename": "SG13_pmos~W0u18_L10u0_S554_1~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u18_L10u0_S554_1~dc_idvg~300K.mdm",\n     "sha256": "eededda2946f9106b9a2813793fd2f48915d9ba7ad6e61cb98292e8826a75622"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_pmos_W0.3um_L10um",\n   "polarity": "pmos",\n   "W_um": 0.3,\n   "L_um": 10.0,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_pmos~W0u3_L10u0_S553_2~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u3_L10u0_S553_2~dc_idvd_vth~300K.mdm",\n     "sha256": "10c58cb32992b67200fcdd21fbfe5b0307e6060bfef1320282fe852d430d049c"\n    },\n    {\n     "filename": "SG13_pmos~W0u3_L10u0_S553_2~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u3_L10u0_S553_2~dc_idvd~300K.mdm",\n     "sha256": "9123f02566a1001a83c0912f26ecce290ae216778f178fe8397b951eb10b1e7b"\n    },\n    {\n     "filename": "SG13_pmos~W0u3_L10u0_S553_2~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u3_L10u0_S553_2~dc_idvg~300K.mdm",\n     "sha256": "a52e2694aae9a0f4db23606275719aaaaa09c219dc7f7a2943288a04bc9639ee"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_pmos_W0.6um_L10um",\n   "polarity": "pmos",\n   "W_um": 0.6,\n   "L_um": 10.0,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_pmos~W0u6_L10u0_S552_1~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u6_L10u0_S552_1~dc_idvd_vth~300K.mdm",\n     "sha256": "5ad8e82b292f265b3ab7ec3f9f2445140923c363ac40ad2ae8fa1caca6ed3137"\n    },\n    {\n     "filename": "SG13_pmos~W0u6_L10u0_S552_1~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u6_L10u0_S552_1~dc_idvd~300K.mdm",\n     "sha256": "500ade1881307ea0bdaf2644c9d220c4283eac76201ced2a5b67d2e24c3420dc"\n    },\n    {\n     "filename": "SG13_pmos~W0u6_L10u0_S552_1~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W0u6_L10u0_S552_1~dc_idvg~300K.mdm",\n     "sha256": "01c25330d82802ed92b45d4ce9d95372279f9f7e7f2c7f04fe60397e14315649"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_pmos_W2um_L10um",\n   "polarity": "pmos",\n   "W_um": 2.0,\n   "L_um": 10.0,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_pmos~W02u0_L10u0_S551_1~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W02u0_L10u0_S551_1~dc_idvd_vth~300K.mdm",\n     "sha256": "dfbd558695e6d61b4265c002348aaaa2d492b037faf144b1a02ee5b06380fc06"\n    },\n    {\n     "filename": "SG13_pmos~W02u0_L10u0_S551_1~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W02u0_L10u0_S551_1~dc_idvd~300K.mdm",\n     "sha256": "2e5c80ee204c356963af2a7a40e951fccd6dc75a528abf99b2074aedb6382393"\n    },\n    {\n     "filename": "SG13_pmos~W02u0_L10u0_S551_1~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W02u0_L10u0_S551_1~dc_idvg~300K.mdm",\n     "sha256": "fd73a5abbba6b4827c529ae79bd6ae5b230920a01dc65e2a876f817186403eb0"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_pmos_W5um_L10um",\n   "polarity": "pmos",\n   "W_um": 5.0,\n   "L_um": 10.0,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_pmos~W05u0_L10u0_S550_3~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W05u0_L10u0_S550_3~dc_idvd_vth~300K.mdm",\n     "sha256": "8f9f5e079b09fb0571d09c98e14f0e6f838c6f4a0ec458330113f9e80f69db00"\n    },\n    {\n     "filename": "SG13_pmos~W05u0_L10u0_S550_3~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W05u0_L10u0_S550_3~dc_idvd~300K.mdm",\n     "sha256": "7be93f7cbe7df097837210b2ff8990d734d509cfe0e55ed5b7e69dd1e9867bdb"\n    },\n    {\n     "filename": "SG13_pmos~W05u0_L10u0_S550_3~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W05u0_L10u0_S550_3~dc_idvg~300K.mdm",\n     "sha256": "24182451d26dcede103f2fa0d3f9a6422f7c00e4a7f4fc1c0cd4a445cb825b71"\n    }\n   ]\n  },\n  {\n   "device_id": "IHP_pmos_W10um_L10um",\n   "polarity": "pmos",\n   "W_um": 10.0,\n   "L_um": 10.0,\n   "temperature_C": 27.0,\n   "files": [\n    {\n     "filename": "SG13_pmos~W10u0_L10u0_S549_5~dc_idvd_vth~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W10u0_L10u0_S549_5~dc_idvd_vth~300K.mdm",\n     "sha256": "27d36a37f3960260854d1060120c36c1bbb8bad20e992604281ab2a5745743a8"\n    },\n    {\n     "filename": "SG13_pmos~W10u0_L10u0_S549_5~dc_idvd~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W10u0_L10u0_S549_5~dc_idvd~300K.mdm",\n     "sha256": "bb12389d5789b8035d388e00253b7d65e5d95d6fdf95d1ee147859b564dd2e47"\n    },\n    {\n     "filename": "SG13_pmos~W10u0_L10u0_S549_5~dc_idvg~300K.mdm",\n     "url": "https://raw.githubusercontent.com/IHP-GmbH/IHP-Open-PDK/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3/SG13_pmos~W10u0_L10u0_S549_5~dc_idvg~300K.mdm",\n     "sha256": "af15a45728fcf7de8f3c4c21819590992709ec50940a19ff54b1f1ac0309ce31"\n    }\n   ]\n  }\n ]\n}', 'csv_template/ihp_curve_00.csv': 'voltage_V,current_A\n0.0,6.474e-11\n0.05,1.6004e-08\n0.1,1.9662e-08\n0.15,2.1392e-08\n0.2,2.2836e-08\n0.25,2.427e-08\n0.3,2.5748e-08\n0.35,2.7272e-08\n0.4,2.8876e-08\n0.45,3.053e-08\n0.5,3.227e-08\n0.55,3.41e-08\n0.6,3.6006e-08\n0.65,3.7986e-08\n0.7,4.0074e-08\n0.75,4.2254e-08\n0.8,4.4532e-08\n0.85,4.6916e-08\n0.9,4.942e-08\n0.95,5.2014e-08\n1.0,5.4762e-08\n1.05,5.7644e-08\n1.1,6.0668e-08\n1.15,6.3856e-08\n1.2,6.7156e-08\n1.25,7.0642e-08\n1.3,7.4294e-08\n1.35,7.8192e-08\n', 'csv_template/ihp_curve_01.csv': 'voltage_V,current_A\n0.0,3.0682e-09\n0.05,3.811e-06\n0.1,4.9678e-06\n0.15,5.5116e-06\n0.2,5.9114e-06\n0.25,6.2894e-06\n0.3,6.6466e-06\n0.35,6.9878e-06\n0.4,7.3494e-06\n0.45,7.7148e-06\n0.5,8.1068e-06\n0.55,8.4944e-06\n0.6,8.8756e-06\n0.65,9.2752e-06\n0.7,9.6856e-06\n0.75,1.01146e-05\n0.8,1.05506e-05\n0.85,1.09968e-05\n0.9,1.1466e-05\n0.95,1.1938e-05\n1.0,1.242e-05\n1.05,1.2924e-05\n1.1,1.3444e-05\n1.15,1.3986e-05\n1.2,1.454e-05\n1.25,1.5126e-05\n1.3,1.5708e-05\n1.35,1.633e-05\n', 'csv_template/ihp_curve_02.csv': 'voltage_V,current_A\n0.0,2.854e-07\n0.05,8.9562e-05\n0.1,0.0001402\n0.15,0.00016836\n0.2,0.00018544\n0.25,0.00019708\n0.3,0.0002064\n0.35,0.00021452\n0.4,0.0002218\n0.45,0.00022882\n0.5,0.00023558\n0.55,0.00024232\n0.6,0.00024888\n0.65,0.00025538\n0.7,0.00026194\n0.75,0.00026852\n0.8,0.00027502\n0.85,0.00028154\n0.9,0.00028804\n0.95,0.00029462\n1.0,0.00030132\n1.05,0.00030836\n1.1,0.00031506\n1.15,0.00032184\n1.2,0.00032868\n1.25,0.0003356\n1.3,0.00034252\n1.35,0.0003495\n', 'csv_template/ihp_curve_03.csv': 'voltage_V,current_A\n0.0,6.8014e-08\n0.05,2.6024e-05\n0.1,3.723e-05\n0.15,4.2506e-05\n0.2,4.587e-05\n0.25,4.856e-05\n0.3,5.1146e-05\n0.35,5.3466e-05\n0.4,5.563e-05\n0.45,5.7984e-05\n0.5,5.999e-05\n0.55,6.246e-05\n0.6,6.4466e-05\n0.65,6.6758e-05\n0.7,6.903e-05\n0.75,7.1286e-05\n0.8,7.3582e-05\n0.85,7.5932e-05\n0.9,7.8468e-05\n0.95,8.0896e-05\n1.0,8.3248e-05\n1.05,8.5886e-05\n1.1,8.8532e-05\n1.15,9.1136e-05\n1.2,9.3986e-05\n1.25,9.644e-05\n1.3,9.9216e-05\n1.35,0.000102326\n', 'csv_template/ihp_curve_04.csv': 'voltage_V,current_A\n0.0,5.8506e-07\n0.05,0.00021262\n0.1,0.00036078\n0.15,0.00045848\n0.2,0.00052186\n0.25,0.0005641\n0.3,0.00059412\n0.35,0.00061722\n0.4,0.00063652\n0.45,0.00065352\n0.5,0.00066918\n0.55,0.00068336\n0.6,0.00069784\n0.65,0.00071162\n0.7,0.00072498\n0.75,0.0007383\n0.8,0.0007513\n0.85,0.00076416\n0.9,0.00077702\n0.95,0.00078978\n1.0,0.00080234\n1.05,0.00081514\n1.1,0.00082768\n1.15,0.00084042\n1.2,0.00085302\n1.25,0.00086558\n1.3,0.00087852\n1.35,0.00089108\n', 'csv_template/ihp_curve_05.csv': 'voltage_V,current_A\n0.0,7.164e-07\n0.05,0.00046624\n0.1,0.00085194\n0.15,0.0011546\n0.2,0.001381\n0.25,0.001547\n0.3,0.0016676\n0.35,0.0017566\n0.4,0.001823\n0.45,0.0018756\n0.5,0.0019184\n0.55,0.001955\n0.6,0.0019878\n0.65,0.0020166\n0.7,0.0020444\n0.75,0.0020702\n0.8,0.0020956\n0.85,0.0021194\n0.9,0.0021428\n0.95,0.0021658\n1.0,0.0021878\n1.05,0.0022098\n1.1,0.0022316\n1.15,0.002253\n1.2,0.002274\n1.25,0.002295\n1.3,0.0023164\n1.35,0.0023372\n', 'csv_template/ihp_curve_06.csv': 'voltage_V,current_A\n0.0,1.058e-06\n0.05,0.00067616\n0.1,0.0012762\n0.15,0.0017824\n0.2,0.0021942\n0.25,0.0025214\n0.3,0.0027774\n0.35,0.0029746\n0.4,0.003128\n0.45,0.0032466\n0.5,0.003341\n0.55,0.0034156\n0.6,0.0034782\n0.65,0.0035322\n0.7,0.0035788\n0.75,0.003621\n0.8,0.003659\n0.85,0.003695\n0.9,0.0037284\n0.95,0.003761\n1.0,0.003792\n1.05,0.0038208\n1.1,0.0038498\n1.15,0.0038776\n1.2,0.0039054\n1.25,0.0039318\n1.3,0.003958\n1.35,0.0039844\n', 'csv_template/ihp_curve_07.csv': 'voltage_V,current_A\n0.0,1.2088e-06\n0.05,0.00082932\n0.1,0.0015918\n0.15,0.0022636\n0.2,0.002838\n0.25,0.0033184\n0.3,0.003713\n0.35,0.004034\n0.4,0.0042938\n0.45,0.0045014\n0.5,0.004669\n0.55,0.0048054\n0.6,0.004917\n0.65,0.0050088\n0.7,0.0050864\n0.75,0.0051526\n0.8,0.005211\n0.85,0.005263\n0.9,0.0053106\n0.95,0.0053548\n1.0,0.0053958\n1.05,0.005434\n1.1,0.005471\n1.15,0.0055062\n1.2,0.0055394\n1.25,0.0055714\n1.3,0.0056028\n1.35,0.0056338\n', 'csv_template/ihp_curve_08.csv': 'voltage_V,current_A\n-0.5,2e-12\n-0.45,1.48e-12\n-0.4,1.14e-12\n-0.35,6.8e-13\n-0.3,5.4e-13\n-0.25,3.8e-13\n-0.2,4.2e-13\n-0.15,1.24e-12\n-0.1,5.22e-12\n-0.05,2.21e-11\n0.0,9.542e-11\n0.05,4.1476e-10\n0.1,1.811e-09\n0.15,7.9092e-09\n0.2,3.419e-08\n0.25,1.4526e-07\n0.3,5.8674e-07\n0.35,2.1488e-06\n0.4,6.7362e-06\n0.45,1.7548e-05\n0.5,3.8074e-05\n0.55,7.0126e-05\n0.6,0.000112996\n0.65,0.00016436\n0.7,0.00022074\n0.75,0.0002797\n0.8,0.00033898\n0.85,0.0003972\n0.9,0.00045338\n0.95,0.00050704\n1.0,0.00055766\n1.05,0.00060528\n1.1,0.00064982\n1.15,0.00069146\n1.2,0.00073008\n1.25,0.0007658\n1.3,0.00079902\n1.35,0.00082956\n', 'csv_template/ihp_curve_09.csv': 'voltage_V,current_A\n-0.5,2.238e-11\n-0.45,1.858e-11\n-0.4,1.552e-11\n-0.35,1.302e-11\n-0.3,1.076e-11\n-0.25,8.84e-12\n-0.2,7.66e-12\n-0.15,8.3e-12\n-0.1,1.492e-11\n-0.05,4.934e-11\n0.0,2.04e-10\n0.05,8.9592e-10\n0.1,3.9718e-09\n0.15,1.77e-08\n0.2,7.7818e-08\n0.25,3.3502e-07\n0.3,1.3618e-06\n0.35,4.984e-06\n0.4,1.5892e-05\n0.45,4.2666e-05\n0.5,9.682e-05\n0.55,0.00018896\n0.6,0.00032608\n0.65,0.00050848\n0.7,0.00073182\n0.75,0.00099064\n0.8,0.001278\n0.85,0.0015858\n0.9,0.0019086\n0.95,0.002242\n1.0,0.0025822\n1.05,0.0029254\n1.1,0.0032678\n1.15,0.0036088\n1.2,0.0039456\n1.25,0.0042772\n1.3,0.0046018\n1.35,0.0049176\n', 'csv_template/ihp_curve_10.csv': 'voltage_V,current_A\n-0.5,1.1974e-10\n-0.45,1.0382e-10\n-0.4,9.096e-11\n-0.35,7.978e-11\n-0.3,7.026e-11\n-0.25,6.162e-11\n-0.2,5.422e-11\n-0.15,5.07e-11\n-0.1,5.854e-11\n-0.05,1.1596e-10\n0.0,3.9312e-10\n0.05,1.6612e-09\n0.1,7.368e-09\n0.15,3.2916e-08\n0.2,1.4472e-07\n0.25,6.1438e-07\n0.3,2.405e-06\n0.35,8.3878e-06\n0.4,2.5198e-05\n0.45,6.3882e-05\n0.5,0.00013706\n0.55,0.00025454\n0.6,0.0004212\n0.65,0.00063544\n0.7,0.00089134\n0.75,0.0011824\n0.8,0.0014992\n0.85,0.001837\n0.9,0.002189\n0.95,0.0025518\n1.0,0.002922\n1.05,0.0032966\n1.1,0.0036724\n1.15,0.0040506\n1.2,0.004427\n1.25,0.0048012\n1.3,0.0051724\n1.35,0.0055408\n', 'csv_template/manifest.csv': 'device_id,file,sweep,fixed_voltage_V,x_column,current_column,W_um,L_um,temperature_C,m,voltage_scale,current_scale,separator,split\ncustom_IHP_W10_L013,ihp_curve_00.csv,idvd,0.174,voltage_V,current_A,10.0,0.13,27.0,1.0,1.0,1.0,comma,fit\ncustom_IHP_W10_L013,ihp_curve_01.csv,idvd,0.374,voltage_V,current_A,10.0,0.13,27.0,1.0,1.0,1.0,comma,fit\ncustom_IHP_W10_L013,ihp_curve_02.csv,idvd,0.574,voltage_V,current_A,10.0,0.13,27.0,1.0,1.0,1.0,comma,fit\ncustom_IHP_W10_L013,ihp_curve_03.csv,idvd,0.474,voltage_V,current_A,10.0,0.13,27.0,1.0,1.0,1.0,comma,fit\ncustom_IHP_W10_L013,ihp_curve_04.csv,idvd,0.693,voltage_V,current_A,10.0,0.13,27.0,1.0,1.0,1.0,comma,fit\ncustom_IHP_W10_L013,ihp_curve_05.csv,idvd,0.912,voltage_V,current_A,10.0,0.13,27.0,1.0,1.0,1.0,comma,fit\ncustom_IHP_W10_L013,ihp_curve_06.csv,idvd,1.131,voltage_V,current_A,10.0,0.13,27.0,1.0,1.0,1.0,comma,fit\ncustom_IHP_W10_L013,ihp_curve_07.csv,idvd,1.35,voltage_V,current_A,10.0,0.13,27.0,1.0,1.0,1.0,comma,fit\ncustom_IHP_W10_L013,ihp_curve_08.csv,idvg,0.05,voltage_V,current_A,10.0,0.13,27.0,1.0,1.0,1.0,comma,fit\ncustom_IHP_W10_L013,ihp_curve_09.csv,idvg,0.6,voltage_V,current_A,10.0,0.13,27.0,1.0,1.0,1.0,comma,validation\ncustom_IHP_W10_L013,ihp_curve_10.csv,idvg,1.2,voltage_V,current_A,10.0,0.13,27.0,1.0,1.0,1.0,comma,fit\n'}
for name, text in ASSETS.items():
    path = WORK_DIR / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding='utf-8')
print('IHP data definitions and CSV input example are ready.')

## 2. Choose the data and the fitting settings
**IHP data.** Give the device IDs in `DEVICE_IDS` (comma separated); the next cell prints all 76 IDs, of the form
`IHP_nmos_W10um_L0.13um` and `IHP_pmos_W10um_L0.13um`. Each device is fitted independently.

**Your own files.** Upload `manifest.csv` together with every file it names. Files may use comma, semicolon, tab or
space delimiters and need named columns. Each row of `manifest.csv` describes one I–V curve:

| Field | Meaning |
|---|---|
| device_id | a unique name for one device (W, L, temperature, polarity) |
| file | file name beside manifest.csv |
| sweep | `idvg` (gate sweep) or `idvd` (drain sweep) |
| fixed_voltage_V | V_DS of an idvg curve, V_GS of an idvd curve, in volts |
| x_column / current_column | names of the swept-voltage and drain-current columns |
| W_um / L_um / temperature_C / m | drawn geometry, temperature and multiplicity |
| polarity | `nmos` (default) or `pmos` |
| voltage_scale / current_scale | factors to volts and amperes (default 1) |
| separator | auto, comma, semicolon, tab or space |
| split | `fit` or `validation` (default fit) |
| skiprows | optional number of header lines to skip |

Give the drain current **into the drain** and the voltages as measured: for PMOS they are negative; the notebook
maps PMOS data to the NMOS frame (voltages and currents negated) and exports the card with `type=-1`.
Source and bulk are grounded. Each curve needs at least 11 uniformly spaced samples; transfer and output curves are
both required. Do not mix geometries, temperatures or polarities under one device_id.

In [ ]:
#@title Data and optimization
import sys, json, importlib
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))
import acm2_dc, acm2_extraction
importlib.reload(acm2_dc)
importlib.reload(acm2_extraction)
from acm2_dc import DEFAULT_BOUNDS, VECTOR_NAMES
from acm2_extraction import (ihp_dataset, user_dataset, prepare_devices,
                             run_extraction, CURRENT_WEIGHTS, FINAL_WEIGHTS)
from IPython.display import display, Image
import pandas as pd

DATA_SOURCE = "IHP data" #@param ["IHP data", "My I-V files"]
DEVICE_IDS = "IHP_nmos_W10um_L0.13um, IHP_pmos_W10um_L0.13um" #@param {type:"string"}
PROFILE = "standard" #@param ["quick", "standard", "extended"]
SEED_LIST = "11" #@param {type:"string"}
HOLD_OUT_TRANSFER_06V = True #@param {type:"boolean"}
DOWNLOAD_RESULTS = False #@param {type:"boolean"}
SPEC = json.loads((WORK_DIR / "ihp_examples.json").read_text())
AVAILABLE = [d["device_id"] for d in SPEC["devices"]]
SELECTED_DEVICE_IDS = [x.strip() for x in DEVICE_IDS.split(",") if x.strip()]
SEEDS = [int(value.strip()) for value in SEED_LIST.split(",") if value.strip()]
BOUNDS = list(DEFAULT_BOUNDS)
WEIGHTS = dict(FINAL_WEIGHTS)
CURRENT_FLOOR_A = 1e-13
# Local Jupyter only; in Colab an upload dialog opens.
USER_MANIFEST = WORK_DIR / "csv_template/manifest.csv"
print("Available IHP device IDs:")
print(", ".join(AVAILABLE))
display(pd.DataFrame(BOUNDS, index=VECTOR_NAMES, columns=["lower", "upper"]))
print("The optimizer searches log10 of the device specific current I_S; the card reports ish = I_S*L/W.")
print("Derivative-weighted stage:", WEIGHTS)

In [ ]:
#@title Optional: download the CSV input example
DOWNLOAD_CSV_TEMPLATE = False #@param {type:"boolean"}
if DOWNLOAD_CSV_TEMPLATE:
    import shutil
    template_zip = shutil.make_archive(str(WORK_DIR/"CSV_input_example"), "zip", WORK_DIR/"csv_template")
    try:
        from google.colab import files
        files.download(template_zip)
    except ImportError:
        print(template_zip)

In [ ]:
from datetime import datetime, timezone
if DATA_SOURCE == "IHP data":
    INPUT_FRAME, INPUT_SOURCES = ihp_dataset(SPEC, SELECTED_DEVICE_IDS, WORK_DIR/"ihp_cache", HOLD_OUT_TRANSFER_06V)
else:
    try:
        from google.colab import files
    except ImportError:
        if not Path(USER_MANIFEST).is_file():
            raise FileNotFoundError("Set USER_MANIFEST to your local manifest.csv.")
    else:
        print("Select manifest.csv AND the I-V files named inside it.")
        uploaded = files.upload()
        upload_dir = WORK_DIR/"uploads"/datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
        upload_dir.mkdir(parents=True)
        for name, data in uploaded.items():
            if Path(name).name != name:
                raise ValueError("Upload files with simple file names, without directory paths.")
            (upload_dir/name).write_bytes(data)
        USER_MANIFEST = upload_dir/"manifest.csv"
    INPUT_FRAME, INPUT_SOURCES = user_dataset(USER_MANIFEST)
DEVICES = prepare_devices(INPUT_FRAME)
inventory = []
for d in DEVICES:
    for c in d.curves:
        inventory.append(dict(device=d.device_id, polarity=d.polarity, W_um=d.W_um, L_um=d.L_um,
            temperature_C=d.temperature_C, sweep=c.sweep, fixed_V=c.fixed_voltage,
            points=len(c.x), step_V=float(c.x[1]-c.x[0]), split=c.split))
display(pd.DataFrame(inventory))
exclusions = [item for d in DEVICES for item in d.excluded_curves]
if exclusions:
    print("Excluded from fitting; raw input retained:")
    display(pd.DataFrame(exclusions))
print("Input rows:", len(INPUT_FRAME), "| Devices:", len(DEVICES), "| PMOS values are shown in the NMOS frame.")

## 3. Extract the five DC parameters
Stage 1: differential evolution (DE) on linear and logarithmic current. Stage 2: DE with gm and g_ds added,
followed by bounded least squares. No initial card is needed. The specific current and zeta are searched on a
logarithmic scale. The bounds and weights above are editable engineering choices, not universal constants.

Squared residuals are averaged within each curve and then across curves, so weights do not grow with the number of
samples. Linear current and derivative residuals are divided by the curve maximum; log-current residuals are in
decades, for currents above CURRENT_FLOOR_A. The same Savitzky–Golay operator is applied to measured and model
currents (9/3 for gm and g_ds, 11/5 for gm2 and gm3). **gm2 and gm3 are shown for validation only; they are not fitted.**
Only V_G > 0 and V_DS > 0 (NMOS frame) enter the objective. Curves marked validation (by default the
V_DS = 0.6 V transfer curve) are excluded from every fitting stage.

Each run creates a new folder. Stopping messages, seeds, parameters near a bound and costs are recorded.
A low fitting error does not prove unique parameter identification or accuracy outside the supplied data.

In [ ]:
RUN_DIR = WORK_DIR/"results"/datetime.now(timezone.utc).strftime("ACM2_%Y%m%d_%H%M%S_%f")
PARAMETERS, METRICS, RESULT_ZIP = run_extraction(
    INPUT_FRAME, INPUT_SOURCES, RUN_DIR, profile=PROFILE, seeds=SEEDS,
    bounds=BOUNDS, weights=WEIGHTS, current_floor=CURRENT_FLOOR_A)
display(PARAMETERS)
print("Saved results:", RUN_DIR)

## 4. Review the fits and the errors
The table gives VT0 (V), **ish (A)**, n, sigma and zeta, with the W and L of the extraction. The specific current of
an instance is I_S = ish·w/l (per multiplicity unit; m multiplies the current).

The error table is the **mean of the per-curve RMSE, in percent of each curve's maximum**, for fitted and held-out
curves; `metrics_all_curves.csv` has every curve. Markers: data; lines: ACM2. PMOS curves are plotted in the NMOS
frame. The ZIP also contains vector PDFs and every plotted sample.

In [ ]:
summary = (METRICS.groupby(["device_id","split","quantity"], sort=False)
           .max_normalized_RMSE_pct.mean().rename("mean_curve_RMSE_pct").reset_index())
display(summary.pivot(index=["device_id","split"], columns="quantity", values="mean_curve_RMSE_pct").round(3))
for device_id in PARAMETERS.device_id:
    display(Image(filename=str(RUN_DIR/device_id/"comparison.png")))

## 5. Download and use the cards
The ZIP contains `acm2_cards.lib` (all cards of the run), one `.lib` card per device, the parameter table, per-curve
errors, all current and derivative samples, the plots, the exact `acm2.va` and Python code, the settings, the input
hashes and the library versions.

Use in ngspice (OSDI):
```
* build once:  openvaf-r acm2.va -o acm2.osdi
.temp 27
.include acm2_cards.lib
N1 d g s b <card name> w=10u l=0.13u
.control
pre_osdi acm2.osdi
...
```
Set `.temp` to the extraction temperature. The cards are **DC only**: capacitances, junctions, gate resistance and
noise stay at the model defaults. For the IHP SG13G2 devices, the repository's `cards/ihp_sg13g2` files also contain
fitted capacitance and junction parameters.

In [ ]:
print("Result archive:", RESULT_ZIP)
print((RUN_DIR/"acm2_cards.lib").read_text()[:1500])
if DOWNLOAD_RESULTS:
    try:
        from google.colab import files
        files.download(str(RESULT_ZIP))
    except ImportError:
        print("Local Jupyter: the ZIP is at the path printed above.")

## References and data
1. D. G. Alves Neto et al., "Design-Oriented Single-Piece 5-DC-Parameter MOSFET Model," *IEEE Access*, vol. 12,
   pp. 87420–87437, 2024, [doi:10.1109/ACCESS.2024.3417316](https://doi.org/10.1109/ACCESS.2024.3417316).
2. IHP-Open-PDK measurements, [SG13_nmosXm1Y3](https://github.com/IHP-GmbH/IHP-Open-PDK/tree/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_nmosXm1Y3)
   and [SG13_pmosXm1Y3](https://github.com/IHP-GmbH/IHP-Open-PDK/tree/5e6d592e4002946a4616f798c357f0f3c06cf3b6/ihp-sg13g2/libs.doc/meas/MOS/SG13_pmosXm1Y3),
   commit 5e6d592e; every file hash is in ihp_examples.json. IHP-Open-PDK is distributed under the Apache License 2.0;
   the data remain attributed to IHP.
3. [SciPy differential evolution](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.differential_evolution.html).

The Python current is that of `acm2.va` (ACM2 2.0.0) at V_S = V_B = 0 and T = tref.